# **DATA PREPARATION NOTEBOOK**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from scipy import sparse
from pathlib import Path
import joblib

In [2]:
train_df = pd.read_csv("../data/application_train.csv")
columns_description_df = pd.read_csv("../data/HomeCredit_columns_description.csv", encoding="latin1")
data_types = pd.read_csv("../outputs/data_understanding/data_types.csv")

## 1. DATA SELECTION

In [3]:
train_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
train_df.shape

(307511, 122)

In [5]:
#Identifier in the dataset
train_df["SK_ID_CURR"]

0         100002
1         100003
2         100004
3         100006
4         100007
           ...  
307506    456251
307507    456252
307508    456253
307509    456254
307510    456255
Name: SK_ID_CURR, Length: 307511, dtype: int64

In [6]:
#Target column in the dataset
train_df["TARGET"]

0         1
1         0
2         0
3         0
4         0
         ..
307506    0
307507    0
307508    0
307509    1
307510    0
Name: TARGET, Length: 307511, dtype: int64

In [7]:
#Predictor columns in the dataset
initial_predictor_df = train_df.copy()
initial_predictor_df = initial_predictor_df.drop(columns=["SK_ID_CURR", "TARGET"])
initial_predictor_df

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,225000.0,Unaccompanied,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,225000.0,Unaccompanied,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,585000.0,Unaccompanied,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,319500.0,Unaccompanied,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
#Initial number of features
initial_predictor_df.shape

(307511, 120)

In [9]:
data_types

,Unnamed: 0,Feature,Pandas Type,Analytical Type
0,0,SK_ID_CURR,Integer,Identifier
1,1,TARGET,Integer,Target
2,2,NAME_CONTRACT_TYPE,str,Binary / Flag
3,3,CODE_GENDER,str,Categorical
4,4,FLAG_OWN_CAR,str,Binary / Flag
...,...,...,...,...
117,117,AMT_REQ_CREDIT_BUREAU_DAY,Float,Numerical Count
118,118,AMT_REQ_CREDIT_BUREAU_WEEK,Float,Numerical Count
119,119,AMT_REQ_CREDIT_BUREAU_MON,Float,Numerical Count
120,120,AMT_REQ_CREDIT_BUREAU_QRT,Float,Numerical Count


In [10]:
data_selection = []

analytical_types = data_types.set_index("Feature")["Analytical Type"]

for feature in train_df.columns:
    if feature == "SK_ID_CURR":
        selected, reason = "No predictor", "Identifier"
    elif feature == "TARGET":
        selected, reason = "Target", "Prediction target"
    else:
        selected, reason = "Yes", "Candidate predictor"

    data_selection.append({
        "Feature": feature,
        "Analytical Type": analytical_types.loc[feature],
        "Selected": selected,
        "Reason": reason,
    })

data_selection_df = pd.DataFrame(data_selection)
data_selection_df.to_csv("../outputs/data_preparation/data_selection.csv")
data_selection_df

,Feature,Analytical Type,Selected,Reason
0,SK_ID_CURR,Identifier,No predictor,Identifier
1,TARGET,Target,Target,Prediction target
2,NAME_CONTRACT_TYPE,Binary / Flag,Yes,Candidate predictor
3,CODE_GENDER,Categorical,Yes,Candidate predictor
4,FLAG_OWN_CAR,Binary / Flag,Yes,Candidate predictor
...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,Numerical Count,Yes,Candidate predictor
118,AMT_REQ_CREDIT_BUREAU_WEEK,Numerical Count,Yes,Candidate predictor
119,AMT_REQ_CREDIT_BUREAU_MON,Numerical Count,Yes,Candidate predictor
120,AMT_REQ_CREDIT_BUREAU_QRT,Numerical Count,Yes,Candidate predictor


## 2. MISSING VALUES

First, calculate the missingness; for each feature with missing values, I must analyze the following options:

- Keep + impute
- Keep + explicit missing category
- Keep + model-native handling
- Drop


Options for handling numerical features:
- Median imputation
- Mean imputation
- Model-native missing handling
- Special/sentinel value


Options for handling categorical features:
- Mode
- “Missing”/“Unknown” category
- Model-native handling

I will treat the features named “EXT_SOURCE_X” specially, since I observed that they can have significant predictive power.

In [11]:
numerical_features = data_types.loc[
    data_types["Analytical Type"].isin([
        "Numerical Continuous",
        "Numerical Count",
        "Temporal / Days",
        "External Score"
    ]),
    "Feature"
].tolist()
numerical_df = train_df[numerical_features]
print(numerical_df.shape)
numerical_df

(307511, 70)


,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637,-3648.0,-2120,...,2.0,2.0,2.0,-1134.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188,-1186.0,-291,...,0.0,1.0,0.0,-828.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225,-4260.0,-2531,...,0.0,0.0,0.0,-815.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039,-9833.0,-2437,...,0.0,2.0,0.0,-617.0,NaN,NaN,NaN,NaN,NaN,NaN
4,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038,-4311.0,-3458,...,0.0,0.0,0.0,-1106.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,0,157500.0,254700.0,27558.0,225000.0,0.032561,-9327,-236,-8456.0,-1982,...,0.0,0.0,0.0,-273.0,NaN,NaN,NaN,NaN,NaN,NaN
307507,0,72000.0,269550.0,12001.5,225000.0,0.025164,-20775,365243,-4388.0,-4090,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
307508,0,153000.0,677664.0,29979.0,585000.0,0.005002,-14966,-7921,-6737.0,-5150,...,0.0,6.0,0.0,-1909.0,1.0,0.0,0.0,1.0,0.0,1.0
307509,0,171000.0,370107.0,20205.0,319500.0,0.005313,-11961,-4786,-2562.0,-931,...,0.0,0.0,0.0,-322.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
categorical_features = data_types.loc[
    data_types["Analytical Type"].isin([
        "Categorical",
        "Ordinal",
        "Binary / Flag"
    ]),
    "Feature"
].tolist()
categorical_df = train_df[categorical_features]
print(categorical_df.shape)
categorical_df

(307511, 50)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,...,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21
0,Cash loans,M,N,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
1,Cash loans,F,N,N,Family,State servant,Higher education,Married,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
2,Revolving loans,M,Y,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
3,Cash loans,F,N,Y,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
4,Cash loans,M,N,Y,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,Cash loans,M,N,N,Unaccompanied,Working,Secondary / secondary special,Separated,With parents,1,...,0,0,0,0,0,0,0,0,0,0
307507,Cash loans,F,N,Y,Unaccompanied,Pensioner,Secondary / secondary special,Widow,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
307508,Cash loans,F,N,Y,Unaccompanied,Working,Higher education,Separated,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0
307509,Cash loans,F,N,Y,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,1,...,0,0,0,0,0,0,0,0,0,0


In [13]:
#Treatment of missing values per feature.
methods_missing_values = {
    "Categorical": "Missing / Unknown",
    "Binary / Flag": "Moda",
    "Numerical Continuous": "Median",
    "Numerical Count": "Median",
    "Temporal / Days": "Median",
    "Ordinal": "Unknown",
    "External Score": "Median",
}

critical_missing_values = ["LOW", "MEDIUM", "HIGH"]

In [14]:
missing_values_general = {}

for i in train_df:
    count_null_values = train_df[i].isnull().sum()
    count_total_values = train_df[i].shape[0]
    null_percentage = (count_null_values / count_total_values) * 100
    missing_values_general[i] = {
        "Missing Count": count_null_values,
        "Missing Percentage": null_percentage
    }

missing_values_per_feature_df = pd.DataFrame(data=missing_values_general).sort_values(by="Missing Percentage", axis=1, ascending=False)
missing_values_per_feature_df.T

,Missing Count,Missing Percentage
COMMONAREA_MEDI,214865.0,69.872297
COMMONAREA_AVG,214865.0,69.872297
COMMONAREA_MODE,214865.0,69.872297
NONLIVINGAPARTMENTS_MODE,213514.0,69.432963
NONLIVINGAPARTMENTS_AVG,213514.0,69.432963
...,...,...
NAME_HOUSING_TYPE,0.0,0.000000
NAME_FAMILY_STATUS,0.0,0.000000
NAME_EDUCATION_TYPE,0.0,0.000000
NAME_INCOME_TYPE,0.0,0.000000


In [15]:
missing_values_per_feature_df = missing_values_per_feature_df.drop(columns=["TARGET", "SK_ID_CURR"])
missing_values_per_feature_df

,COMMONAREA_MEDI,COMMONAREA_AVG,COMMONAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAPARTMENTS_AVG,NONLIVINGAPARTMENTS_MEDI,FONDKAPREMONT_MODE,LIVINGAPARTMENTS_MODE,LIVINGAPARTMENTS_AVG,LIVINGAPARTMENTS_MEDI,...,FLAG_MOBIL,DAYS_ID_PUBLISH,DAYS_REGISTRATION,DAYS_EMPLOYED,DAYS_BIRTH,REGION_POPULATION_RELATIVE,NAME_HOUSING_TYPE,NAME_FAMILY_STATUS,NAME_EDUCATION_TYPE,NAME_INCOME_TYPE
Missing Count,214865.000000,214865.000000,214865.000000,213514.000000,213514.000000,213514.000000,210295.000000,210199.000000,210199.000000,210199.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Missing Percentage,69.872297,69.872297,69.872297,69.432963,69.432963,69.432963,68.386172,68.354953,68.354953,68.354953,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
missing_values_initial_data_methods = {}

for i in missing_values_per_feature_df.columns:
    if missing_values_per_feature_df[i]["Missing Percentage"] <= 40:
        rate = "LOW"
    if missing_values_per_feature_df[i]["Missing Percentage"] > 40 and missing_values_per_feature_df[i]["Missing Percentage"] <= 70:
        rate = "MEDIUM"
    if missing_values_per_feature_df[i]["Missing Percentage"] > 70:
        rate = "HIGH"

    analytical_type_feature =  analytical_types.loc[i]
    data = {
        "Feature": i,
        "Missing %": missing_values_per_feature_df[i]["Missing Percentage"],
        "Rate": rate,
        "Method Apply": methods_missing_values[analytical_type_feature],
    }
    missing_values_initial_data_methods[i] = data

missing_values_initial_data_methods_df = pd.DataFrame.from_dict(
    missing_values_initial_data_methods,
    orient="index"
).reset_index(drop=True)
missing_values_initial_data_methods_df

,Feature,Missing %,Rate,Method Apply
0,COMMONAREA_MEDI,69.872297,MEDIUM,Median
1,COMMONAREA_AVG,69.872297,MEDIUM,Median
2,COMMONAREA_MODE,69.872297,MEDIUM,Median
3,NONLIVINGAPARTMENTS_MODE,69.432963,MEDIUM,Median
4,NONLIVINGAPARTMENTS_AVG,69.432963,MEDIUM,Median
...,...,...,...,...
115,REGION_POPULATION_RELATIVE,0.000000,LOW,Median
116,NAME_HOUSING_TYPE,0.000000,LOW,Missing / Unknown
117,NAME_FAMILY_STATUS,0.000000,LOW,Missing / Unknown
118,NAME_EDUCATION_TYPE,0.000000,LOW,Unknown


In [17]:
data_selection_df = data_selection_df.merge(
    missing_values_initial_data_methods_df,
    on="Feature",
    how="left",
    validate="one_to_one"
)
data_selection_df.sort_values(by="Missing %", ascending=False)

,Feature,Analytical Type,Selected,Reason,Missing %,Rate,Method Apply
62,COMMONAREA_MODE,Numerical Continuous,Yes,Candidate predictor,69.872297,MEDIUM,Median
76,COMMONAREA_MEDI,Numerical Continuous,Yes,Candidate predictor,69.872297,MEDIUM,Median
48,COMMONAREA_AVG,Numerical Continuous,Yes,Candidate predictor,69.872297,MEDIUM,Median
84,NONLIVINGAPARTMENTS_MEDI,Numerical Continuous,Yes,Candidate predictor,69.432963,MEDIUM,Median
56,NONLIVINGAPARTMENTS_AVG,Numerical Continuous,Yes,Candidate predictor,69.432963,MEDIUM,Median
...,...,...,...,...,...,...,...
37,REG_CITY_NOT_LIVE_CITY,Binary / Flag,Yes,Candidate predictor,0.000000,LOW,Moda
38,REG_CITY_NOT_WORK_CITY,Binary / Flag,Yes,Candidate predictor,0.000000,LOW,Moda
2,NAME_CONTRACT_TYPE,Binary / Flag,Yes,Candidate predictor,0.000000,LOW,Moda
0,SK_ID_CURR,Identifier,No predictor,Identifier,NaN,NaN,NaN


In [18]:
data_selection_df.to_csv("../outputs/data_preparation/missing_values_strategy.csv")

## 3. DUPLICATES

There were no duplicate rows or values in our dataset, as we saw in the previous section of the “Data Understanding” notebook.

## 4. OUTLIERS AND ANOMALIES

I'm working on the `prepared_df` dataframe, which is a copy of `train_df`. I'll be performing all the feature engineering on this `prepared_df` to keep the original intact.

In [19]:
prepared_train_df = train_df.copy()
print("Original dataset shape:", train_df.shape)
print("Prepared dataset shape:", prepared_train_df.shape)

Original dataset shape: (307511, 122)
Prepared dataset shape: (307511, 122)


4.1 Review of previously identified anomalies

In [20]:
numerical_features = data_types.loc[
    data_types["Analytical Type"].isin([
        "Numerical Continuous",
        "Numerical Count",
        "Temporal / Days",
        "External Score"
    ]),
    "Feature"
].tolist()

In [21]:
numerical_summary = prepared_train_df[numerical_features].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
).T

In [22]:
anomaly_summary = numerical_summary[["min", "1%", "50%", "99%", "max"]].copy()
anomaly_summary["Max / P99"] = anomaly_summary["max"] / anomaly_summary["99%"].replace(0, np.nan)
anomaly_summary = anomaly_summary.sort_values(by="Max / P99", ascending=False)

top_20_extreme_features = anomaly_summary.head(20)
top_20_extreme_features

,min,1%,50%,99%,max,Max / P99
AMT_INCOME_TOTAL,25650.0,45000.00,147150.0000,472500.000000,117000000.0,247.619048
AMT_REQ_CREDIT_BUREAU_QRT,0.0,0.00,0.0000,2.000000,261.0,130.500000
OBS_30_CNT_SOCIAL_CIRCLE,0.0,0.00,0.0000,10.000000,348.0,34.800000
OBS_60_CNT_SOCIAL_CIRCLE,0.0,0.00,0.0000,10.000000,344.0,34.400000
DEF_30_CNT_SOCIAL_CIRCLE,0.0,0.00,0.0000,2.000000,34.0,17.000000
DEF_60_CNT_SOCIAL_CIRCLE,0.0,0.00,0.0000,2.000000,24.0,12.000000
NONLIVINGAPARTMENTS_MODE,0.0,0.00,0.0000,0.097300,1.0,10.277492
NONLIVINGAPARTMENTS_MEDI,0.0,0.00,0.0000,0.104800,1.0,9.541985
NONLIVINGAPARTMENTS_AVG,0.0,0.00,0.0000,0.108100,1.0,9.250694
AMT_REQ_CREDIT_BUREAU_WEEK,0.0,0.00,0.0000,1.000000,8.0,8.000000


In [23]:
top_20_extreme_features.to_csv("../outputs/data_preparation/outlier_screening_top20.csv")

4.2 Known Sentinel Values

In [24]:
days_employed_sentinel_count = (prepared_train_df["DAYS_EMPLOYED"] == 365243).sum()
days_employed_sentinel_percentage = days_employed_sentinel_count / len(prepared_train_df) * 100

print("DAYS_EMPLOYED sentinel count:", days_employed_sentinel_count)
print("DAYS_EMPLOYED sentinel percentage:", round(days_employed_sentinel_percentage, 2), "%")

DAYS_EMPLOYED sentinel count: 55374
DAYS_EMPLOYED sentinel percentage: 18.01 %


In [25]:
prepared_train_df["DAYS_EMPLOYED_ANOM"] = (prepared_train_df["DAYS_EMPLOYED"] == 365243).astype(int)

In [26]:
prepared_train_df.loc[prepared_train_df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

In [27]:
print("Sentinel values remaining:", (prepared_train_df["DAYS_EMPLOYED"] == 365243).sum())
print("Anomaly flags created:", prepared_train_df["DAYS_EMPLOYED_ANOM"].sum())

Sentinel values remaining: 0
Anomaly flags created: 55374


4.3 Sentinel Value Treatment

In [28]:
bounded_features = [
    "NONLIVINGAPARTMENTS_MODE",
    "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAREA_AVG",
    "NONLIVINGAREA_MEDI",
    "NONLIVINGAREA_MODE",
    "COMMONAREA_MODE",
    "LANDAREA_AVG"
]

In [29]:
bounded_review = pd.DataFrame({
    "Min": prepared_train_df[bounded_features].min(),
    "Max": prepared_train_df[bounded_features].max()
})

bounded_review["Within 0-1 Range"] = ((bounded_review["Min"] >= 0) & (bounded_review["Max"] <= 1))
bounded_review

,Min,Max,Within 0-1 Range
NONLIVINGAPARTMENTS_MODE,0.0,1.0,True
NONLIVINGAPARTMENTS_MEDI,0.0,1.0,True
NONLIVINGAPARTMENTS_AVG,0.0,1.0,True
NONLIVINGAREA_AVG,0.0,1.0,True
NONLIVINGAREA_MEDI,0.0,1.0,True
NONLIVINGAREA_MODE,0.0,1.0,True
COMMONAREA_MODE,0.0,1.0,True
LANDAREA_AVG,0.0,1.0,True


Since these features fall within the range of 0 to 1, I leave them as is; therefore, the treatment to be applied will be treatment=keep

4.4 Monetary Outliers

In [30]:
monetary_outlier_features = [
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY"
]

In [31]:
monetary_review = prepared_train_df[monetary_outlier_features].describe(percentiles=[0.95,0.99,0.999]).T
monetary_review

,count,mean,std,min,95%,99%,99.9%,max
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,337500.0,472500.0,900000.0,117000000.0
AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.5,53325.0,70006.5,110047.5,258025.5


In [32]:
for feature in monetary_outlier_features:
    print(f"\n--- {feature} ---")
    display(prepared_train_df[["SK_ID_CURR",feature]].sort_values(feature,ascending=False).head(10))


--- AMT_INCOME_TOTAL ---


,SK_ID_CURR,AMT_INCOME_TOTAL
12840,114967,117000000.0
203693,336147,18000090.0
246858,385674,13500000.0
77768,190160,9000000.0
131127,252084,6750000.0
103006,219563,4500000.0
204564,337151,4500000.0
187833,317748,4500000.0
287463,432980,4500000.0
181698,310601,3950059.5



--- AMT_ANNUITY ---


,SK_ID_CURR,AMT_ANNUITY
17948,120926,258025.5
101508,217854,230161.5
157614,282697,225000.0
141371,263908,225000.0
266148,408308,225000.0
20119,123467,225000.0
265026,406901,225000.0
104938,221773,225000.0
103006,219563,225000.0
32100,137221,225000.0


4.5 Extreme Count Features

In [33]:
extreme_count_features = [
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "AMT_REQ_CREDIT_BUREAU_YEAR"
]

In [34]:
count_outlier_review = []

for feature in extreme_count_features:
    p99 = prepared_train_df[feature].quantile(0.99)
    maximum = prepared_train_df[feature].max()
    above_p99_count = (prepared_train_df[feature] > p99).sum()
    above_p99_percentage = above_p99_count / prepared_train_df[feature].notna().sum() * 100
    count_outlier_review.append({
        "Feature": feature,
        "P99": p99,
        "Maximum": maximum,
        "Above P99 Count": above_p99_count,
        "Above P99 Percentage": above_p99_percentage
    })

count_outlier_review = pd.DataFrame(count_outlier_review)
count_outlier_review

,Feature,P99,Maximum,Above P99 Count,Above P99 Percentage
0,AMT_REQ_CREDIT_BUREAU_QRT,2.0,261.0,2301,0.865064
1,OBS_30_CNT_SOCIAL_CIRCLE,10.0,348.0,2782,0.907697
2,OBS_60_CNT_SOCIAL_CIRCLE,10.0,344.0,2691,0.878006
3,DEF_30_CNT_SOCIAL_CIRCLE,2.0,34.0,1515,0.494307
4,DEF_60_CNT_SOCIAL_CIRCLE,2.0,24.0,758,0.247316
5,AMT_REQ_CREDIT_BUREAU_WEEK,1.0,8.0,328,0.123312
6,AMT_REQ_CREDIT_BUREAU_MON,4.0,27.0,2159,0.811679
7,CNT_CHILDREN,3.0,19.0,555,0.180481
8,CNT_FAM_MEMBERS,5.0,20.0,529,0.172027
9,AMT_REQ_CREDIT_BUREAU_YEAR,8.0,25.0,1237,0.465052


In [35]:
for feature in extreme_count_features:
    print(f"\n--- {feature} ---")
    display(prepared_train_df[feature].value_counts(dropna=False).sort_index(ascending=False).head(15))


--- AMT_REQ_CREDIT_BUREAU_QRT ---


AMT_REQ_CREDIT_BUREAU_QRT
261.0         1
19.0          1
8.0           7
7.0           7
6.0          28
5.0          64
4.0         476
3.0        1717
2.0       14412
1.0       33862
0.0      215417
NaN       41519
Name: count, dtype: int64


--- OBS_30_CNT_SOCIAL_CIRCLE ---


OBS_30_CNT_SOCIAL_CIRCLE
348.0     1
47.0      1
30.0      2
29.0      1
28.0      1
27.0      5
26.0      3
25.0     11
24.0     11
23.0     15
22.0     22
21.0     29
20.0     30
19.0     44
18.0     46
Name: count, dtype: int64


--- OBS_60_CNT_SOCIAL_CIRCLE ---


OBS_60_CNT_SOCIAL_CIRCLE
344.0     1
47.0      1
30.0      1
29.0      2
28.0      1
27.0      5
26.0      1
25.0     11
24.0     12
23.0     13
22.0     20
21.0     28
20.0     30
19.0     44
18.0     49
Name: count, dtype: int64

... (10 outputs left out)


--- CNT_FAM_MEMBERS ---


CNT_FAM_MEMBERS
20.0        2
16.0        2
15.0        1
14.0        2
13.0        1
12.0        2
11.0        1
10.0        3
9.0         6
8.0        20
7.0        81
6.0       408
5.0      3478
4.0     24697
3.0     52601
Name: count, dtype: int64


--- AMT_REQ_CREDIT_BUREAU_YEAR ---


AMT_REQ_CREDIT_BUREAU_YEAR
25.0     1
23.0     1
22.0     1
21.0     1
20.0     1
19.0     4
18.0     4
17.0     7
16.0     3
15.0     6
14.0    10
13.0    19
12.0    30
11.0    31
10.0    22
Name: count, dtype: int64

For these 10, the treatment is “keep.” There is no evidence in the analysis to suggest that:

- 261 credit inquiries
- 348 social-circle observations
- 19 children
- 20 family members

are invalid codes or values; although they are obviously outliers, this does not allow me to automatically convert them to NaN or delete them.

4.6 Treatment Decisions

In [36]:
outlier_decisions = top_20_extreme_features.reset_index().rename(columns={"index": "Feature"})

In [37]:
def classify_outlier_feature(feature):
    if feature in bounded_features:
        return "Bounded Numerical"
    elif feature in monetary_outlier_features:
        return "Monetary"
    elif feature in extreme_count_features:
        return "Count"
    else:
        return "Other"

In [38]:
outlier_decisions["Outlier Type"] = outlier_decisions["Feature"].apply(classify_outlier_feature)

In [39]:
outlier_decisions["Treatment"] = "Keep"

In [40]:
def assign_outlier_reason(row):
    if row["Outlier Type"] == "Bounded Numerical":
        return "Extreme Max/P99 ratio is caused by the distribution, but values remain within the valid 0-1 range."
    elif row["Outlier Type"] == "Monetary":
        return "Extreme monetary values may represent legitimate applicants; no evidence confirms that they are invalid observations."
    elif row["Outlier Type"] == "Count":
        return "High count values are statistically extreme but no domain rule or sentinel code identifies them as invalid."
    else:
        return "No evidence was found to justify modifying the observation."

In [41]:
outlier_decisions["Reason"] = outlier_decisions.apply(assign_outlier_reason,axis=1)

In [42]:
outlier_decisions["Dataset Modified"] = False

In [43]:
outlier_decisions[
    [
        "Feature",
        "Outlier Type",
        "99%",
        "max",
        "Max / P99",
        "Treatment",
        "Dataset Modified",
        "Reason"
    ]
]

,Feature,Outlier Type,99%,max,Max / P99,Treatment,Dataset Modified,Reason
0,AMT_INCOME_TOTAL,Monetary,472500.000000,117000000.0,247.619048,Keep,False,Extreme monetary values may represent legitima...
1,AMT_REQ_CREDIT_BUREAU_QRT,Count,2.000000,261.0,130.500000,Keep,False,High count values are statistically extreme bu...
2,OBS_30_CNT_SOCIAL_CIRCLE,Count,10.000000,348.0,34.800000,Keep,False,High count values are statistically extreme bu...
3,OBS_60_CNT_SOCIAL_CIRCLE,Count,10.000000,344.0,34.400000,Keep,False,High count values are statistically extreme bu...
4,DEF_30_CNT_SOCIAL_CIRCLE,Count,2.000000,34.0,17.000000,Keep,False,High count values are statistically extreme bu...
5,DEF_60_CNT_SOCIAL_CIRCLE,Count,2.000000,24.0,12.000000,Keep,False,High count values are statistically extreme bu...
6,NONLIVINGAPARTMENTS_MODE,Bounded Numerical,0.097300,1.0,10.277492,Keep,False,Extreme Max/P99 ratio is caused by the distrib...
7,NONLIVINGAPARTMENTS_MEDI,Bounded Numerical,0.104800,1.0,9.541985,Keep,False,Extreme Max/P99 ratio is caused by the distrib...
8,NONLIVINGAPARTMENTS_AVG,Bounded Numerical,0.108100,1.0,9.250694,Keep,False,Extreme Max/P99 ratio is caused by the distrib...
9,AMT_REQ_CREDIT_BUREAU_WEEK,Count,1.000000,8.0,8.000000,Keep,False,High count values are statistically extreme bu...


In [44]:
days_employed_decision = pd.DataFrame({
    "Feature": ["DAYS_EMPLOYED"],
    "Outlier Type": ["Known Sentinel"],
    "99%": [np.nan],
    "max": [365243],
    "Max / P99": [np.nan],
    "Treatment": ["Replace sentinel with NaN and create anomaly indicator"],
    "Dataset Modified": [True],
    "Reason": ["The value 365243 does not represent a realistic employment duration and is treated as a coded sentinel value."]
})

In [45]:
anomaly_treatment_log = pd.concat(
    [
        days_employed_decision,
        outlier_decisions[
            [
                "Feature",
                "Outlier Type",
                "99%",
                "max",
                "Max / P99",
                "Treatment",
                "Dataset Modified",
                "Reason"
            ]
        ]
    ],
    ignore_index=True
)
anomaly_treatment_log

,Feature,Outlier Type,99%,max,Max / P99,Treatment,Dataset Modified,Reason
0,DAYS_EMPLOYED,Known Sentinel,NaN,365243.0,NaN,Replace sentinel with NaN and create anomaly i...,True,The value 365243 does not represent a realisti...
1,AMT_INCOME_TOTAL,Monetary,472500.000000,117000000.0,247.619048,Keep,False,Extreme monetary values may represent legitima...
2,AMT_REQ_CREDIT_BUREAU_QRT,Count,2.000000,261.0,130.500000,Keep,False,High count values are statistically extreme bu...
3,OBS_30_CNT_SOCIAL_CIRCLE,Count,10.000000,348.0,34.800000,Keep,False,High count values are statistically extreme bu...
4,OBS_60_CNT_SOCIAL_CIRCLE,Count,10.000000,344.0,34.400000,Keep,False,High count values are statistically extreme bu...
5,DEF_30_CNT_SOCIAL_CIRCLE,Count,2.000000,34.0,17.000000,Keep,False,High count values are statistically extreme bu...
6,DEF_60_CNT_SOCIAL_CIRCLE,Count,2.000000,24.0,12.000000,Keep,False,High count values are statistically extreme bu...
7,NONLIVINGAPARTMENTS_MODE,Bounded Numerical,0.097300,1.0,10.277492,Keep,False,Extreme Max/P99 ratio is caused by the distrib...
8,NONLIVINGAPARTMENTS_MEDI,Bounded Numerical,0.104800,1.0,9.541985,Keep,False,Extreme Max/P99 ratio is caused by the distrib...
9,NONLIVINGAPARTMENTS_AVG,Bounded Numerical,0.108100,1.0,9.250694,Keep,False,Extreme Max/P99 ratio is caused by the distrib...


4.7 Treatment Verification

In [46]:
print("Original rows:", len(train_df))
print("Prepared rows:", len(prepared_train_df))

Original rows: 307511
Prepared rows: 307511


In [47]:
print("DAYS_EMPLOYED sentinel remaining:",(prepared_train_df["DAYS_EMPLOYED"] == 365243).sum())

DAYS_EMPLOYED sentinel remaining: 0


In [48]:
print("DAYS_EMPLOYED_ANOM = 1:", prepared_train_df["DAYS_EMPLOYED_ANOM"].sum())

DAYS_EMPLOYED_ANOM = 1: 55374


4.8 Export Results

In [49]:
anomaly_treatment_log.to_csv("../outputs/data_preparation/anomaly_treatment.csv",index=False)

In [50]:
count_outlier_review.to_csv("../outputs/data_preparation/extreme_count_review.csv",index=False)

## 5. ENCODING

5.1 Identify Features Requiring Encoding

In [51]:
data_types

,Unnamed: 0,Feature,Pandas Type,Analytical Type
0,0,SK_ID_CURR,Integer,Identifier
1,1,TARGET,Integer,Target
2,2,NAME_CONTRACT_TYPE,str,Binary / Flag
3,3,CODE_GENDER,str,Categorical
4,4,FLAG_OWN_CAR,str,Binary / Flag
...,...,...,...,...
117,117,AMT_REQ_CREDIT_BUREAU_DAY,Float,Numerical Count
118,118,AMT_REQ_CREDIT_BUREAU_WEEK,Float,Numerical Count
119,119,AMT_REQ_CREDIT_BUREAU_MON,Float,Numerical Count
120,120,AMT_REQ_CREDIT_BUREAU_QRT,Float,Numerical Count


5.2 Binary / Flag Encoding Strategy

In [52]:
binary_flag_features = data_types.loc[data_types["Analytical Type"] == "Binary / Flag", "Feature"].tolist()
binary_flag_features

['NAME_CONTRACT_TYPE',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'EMERGENCYSTATE_MODE',
 'FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_3',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_5',
 'FLAG_DOCUMENT_6',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_8',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_10',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_15',
 'FLAG_DOCUMENT_16',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_21']

In [53]:
#Change de binary features from N-Y to 0-1
binary_mapping = {
    "Y": 1,
    "N": 0,
    "Yes": 1,
    "No": 0,
}

for feature in binary_flag_features:
    prepared_train_df[feature] = prepared_train_df[feature].replace(binary_mapping)

5.3 Nominal Categorical Encoding Strategy

One-hot encoding must be applied to the categorical variable later in the pipeline.

5.4 Ordinal Encoding Strategy

In [54]:
ordinal_features = data_types.loc[data_types["Analytical Type"] == "Ordinal","Feature"].tolist()
ordinal_features

['NAME_EDUCATION_TYPE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']

In [55]:
ordinal_feature_types = pd.DataFrame({
    "Feature": ordinal_features,
    "Current Dtype": [prepared_train_df[feature].dtype for feature in ordinal_features]
})
ordinal_feature_types

,Feature,Current Dtype
0,NAME_EDUCATION_TYPE,str
1,REGION_RATING_CLIENT,int64
2,REGION_RATING_CLIENT_W_CITY,int64


In [56]:
for feature in ordinal_features:
    print(f"\n--- {feature} ---")
    print(prepared_train_df[feature].value_counts(dropna=False))


--- NAME_EDUCATION_TYPE ---
NAME_EDUCATION_TYPE
Secondary / secondary special    218391
Higher education                  74863
Incomplete higher                 10277
Lower secondary                    3816
Academic degree                     164
Name: count, dtype: int64

--- REGION_RATING_CLIENT ---
REGION_RATING_CLIENT
2    226984
3     48330
1     32197
Name: count, dtype: int64

--- REGION_RATING_CLIENT_W_CITY ---
REGION_RATING_CLIENT_W_CITY
2    229484
3     43860
1     34167
Name: count, dtype: int64


In [57]:
education_mapping = {
    "Lower secondary": 0,
    "Secondary / secondary special": 1,
    "Incomplete higher": 2,
    "Higher education": 3,
    "Academic degree": 4
}

In [58]:
numeric_ordinal_features = [feature for feature in ordinal_features if pd.api.types.is_numeric_dtype(prepared_train_df[feature])]
numeric_ordinal_features

['REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']

These features should remain numerical as they are; they should not be one-hot encoded.

In [59]:
text_ordinal_features = [feature for feature in ordinal_features if not pd.api.types.is_numeric_dtype(prepared_train_df[feature])]
text_ordinal_features

['NAME_EDUCATION_TYPE']

5.6 Encoding Decision Log

In [60]:
categorical_features = data_types.loc[data_types["Analytical Type"] == "Categorical","Feature"].tolist()
binary_features = data_types.loc[data_types["Analytical Type"] == "Binary / Flag","Feature"].tolist()
ordinal_features = data_types.loc[data_types["Analytical Type"] == "Ordinal","Feature"].tolist()

In [61]:
encoding_strategy = []

for feature in categorical_features + binary_features + ordinal_features:
    analytical_type = data_types.loc[data_types["Feature"] == feature, "Analytical Type"].iloc[0]
    current_dtype = str(prepared_train_df[feature].dtype)
    
    if analytical_type == "Categorical":
        strategy = "One-Hot Encoding"
        applied_now = False
        reason = "Nominal categorical feature with no intrinsic order."
        unknown_strategy = "Ignore unseen categories"
    elif analytical_type == "Binary / Flag":
        if pd.api.types.is_numeric_dtype(prepared_train_df[feature]):
            strategy = "Keep binary numeric encoding"
            reason = "Feature is already represented numerically."
        else:
            strategy = "Binary mapping"
            reason = ("Binary text feature will be converted to a 0/1 representation.")
        applied_now = False
        unknown_strategy = ("Unexpected categories treated as missing")
    elif analytical_type == "Ordinal":
        if pd.api.types.is_numeric_dtype(prepared_train_df[feature]):
            strategy = "Keep existing ordinal encoding"
            reason = "Feature already has a numerical representation preserving its order."
        else:
            strategy = "Ordinal Encoding"
            reason = "Feature has an intrinsic ordering that should be preserved."

        applied_now = False
        unknown_strategy = "Unknown ordinal category encoded separately"

    encoding_strategy.append({
        "Feature": feature,
        "Analytical Type": analytical_type,
        "Current Dtype": current_dtype,
        "Encoding Strategy": strategy,
        "Unknown Category Strategy": unknown_strategy,
        "Applied Now": applied_now,
        "Reason": reason
    })

In [62]:
encoding_strategy = pd.DataFrame(encoding_strategy)
encoding_strategy

,Feature,Analytical Type,Current Dtype,Encoding Strategy,Unknown Category Strategy,Applied Now,Reason
0,CODE_GENDER,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
1,NAME_TYPE_SUITE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
2,NAME_INCOME_TYPE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
3,NAME_FAMILY_STATUS,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
4,NAME_HOUSING_TYPE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
5,OCCUPATION_TYPE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
6,WEEKDAY_APPR_PROCESS_START,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
7,ORGANIZATION_TYPE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
8,FONDKAPREMONT_MODE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...
9,HOUSETYPE_MODE,Categorical,str,One-Hot Encoding,Ignore unseen categories,False,Nominal categorical feature with no intrinsic ...


In [63]:
#Review Binary / Flag
encoding_strategy[encoding_strategy["Analytical Type"] == "Binary / Flag"][
    [
        "Feature",
        "Current Dtype",
        "Encoding Strategy"
    ]
]

,Feature,Current Dtype,Encoding Strategy
11,NAME_CONTRACT_TYPE,str,Binary mapping
12,FLAG_OWN_CAR,object,Binary mapping
13,FLAG_OWN_REALTY,object,Binary mapping
14,FLAG_MOBIL,int64,Keep binary numeric encoding
15,FLAG_EMP_PHONE,int64,Keep binary numeric encoding
16,FLAG_WORK_PHONE,int64,Keep binary numeric encoding
17,FLAG_CONT_MOBILE,int64,Keep binary numeric encoding
18,FLAG_PHONE,int64,Keep binary numeric encoding
19,FLAG_EMAIL,int64,Keep binary numeric encoding
20,REG_REGION_NOT_LIVE_REGION,int64,Keep binary numeric encoding


In [64]:
#Review Ordinal
encoding_strategy[encoding_strategy["Analytical Type"] == "Ordinal"][
    [
        "Feature",
        "Current Dtype",
        "Encoding Strategy"
    ]
]

,Feature,Current Dtype,Encoding Strategy
47,NAME_EDUCATION_TYPE,str,Ordinal Encoding
48,REGION_RATING_CLIENT,int64,Keep existing ordinal encoding
49,REGION_RATING_CLIENT_W_CITY,int64,Keep existing ordinal encoding


In [65]:
encoding_strategy.to_csv("../outputs/data_preparation/encoding_strategy.csv", index=False)

## 6. SCALING AND TRANSFORMATION

6.1 Numerical Feature Identification

In [66]:
continuous_features = data_types.loc[data_types["Analytical Type"] == "Numerical Continuous","Feature"].tolist()
count_features = data_types.loc[data_types["Analytical Type"] == "Numerical Count","Feature"].tolist()
temporal_features = data_types.loc[data_types["Analytical Type"] == "Temporal / Days","Feature"].tolist()
external_score_features = data_types.loc[data_types["Analytical Type"] == "External Score","Feature"].tolist()
scaling_candidate_features = (continuous_features + count_features + temporal_features + external_score_features)

In [67]:
print("Numerical Continuous:", len(continuous_features))
print("Numerical Count:", len(count_features))
print("Temporal / Days:", len(temporal_features))
print("External Score:", len(external_score_features))
print("Total candidates:", len(scaling_candidate_features))

Numerical Continuous: 49
Numerical Count: 12
Temporal / Days: 6
External Score: 3
Total candidates: 70


6.2 Distribution and Skewness Review

Skewness ≈ 0
→ approximately symmetric distribution

Skewness > 0
→ right-skewed

Skewness < 0
→ left-skewed


|skewness| < 0.5     → Low

0.5–1               → Moderate

|skewness| > 1                 → High

In [68]:
skewness_summary = prepared_train_df[scaling_candidate_features].skew(numeric_only=True).sort_values(key=abs,ascending=False).reset_index()
skewness_summary.columns = ["Feature", "Skewness"]
skewness_summary["Absolute Skewness"] = skewness_summary["Skewness"].abs()

In [69]:
def classify_skewness(value):
    value = abs(value)
    if value < 0.5:
        return "Low"
    elif value <= 1:
        return "Moderate"
    else:
        return "High"

In [70]:
skewness_summary["Skewness Level"] = skewness_summary["Skewness"].apply(classify_skewness)

In [71]:
skewness_summary.head(20)

,Feature,Skewness,Absolute Skewness,Skewness Level
0,AMT_INCOME_TOTAL,391.559654,391.559654,High
1,AMT_REQ_CREDIT_BUREAU_QRT,134.365776,134.365776,High
2,AMT_REQ_CREDIT_BUREAU_DAY,27.043505,27.043505,High
3,NONLIVINGAPARTMENTS_MODE,16.251819,16.251819,High
4,NONLIVINGAPARTMENTS_MEDI,15.671995,15.671995,High
5,YEARS_BEGINEXPLUATATION_MEDI,-15.573124,15.573124,High
6,NONLIVINGAPARTMENTS_AVG,15.541185,15.541185,High
7,YEARS_BEGINEXPLUATATION_AVG,-15.515264,15.515264,High
8,YEARS_BEGINEXPLUATATION_MODE,-14.755318,14.755318,High
9,AMT_REQ_CREDIT_BUREAU_HOUR,14.534062,14.534062,High


In [72]:
skewness_summary.to_csv("../outputs/data_preparation/skewness_summary.csv",index=False)

6.3 Transformation Strategy for Skewed Features

In [73]:
#Candidates for log transformation. We need to identify non-negative continuous variables with high skewness
high_skew_continuous = skewness_summary[(skewness_summary["Feature"].isin(continuous_features)) & (skewness_summary["Absolute Skewness"] > 1)]["Feature"].tolist()

In [74]:
log_transform_candidates = [feature for feature in high_skew_continuous if prepared_train_df[feature].min(skipna=True) >= 0]
log_transform_candidates

['AMT_INCOME_TOTAL',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_MEDI',
 'YEARS_BEGINEXPLUATATION_MEDI',
 'NONLIVINGAPARTMENTS_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BEGINEXPLUATATION_MODE',
 'NONLIVINGAREA_AVG',
 'NONLIVINGAREA_MODE',
 'NONLIVINGAREA_MEDI',
 'COMMONAREA_MODE',
 'COMMONAREA_AVG',
 'COMMONAREA_MEDI',
 'LANDAREA_AVG',
 'LANDAREA_MODE',
 'LANDAREA_MEDI',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_MODE',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAPARTMENTS_MEDI',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAREA_MODE',
 'LIVINGAREA_AVG',
 'LIVINGAREA_MEDI',
 'TOTALAREA_MODE',
 'APARTMENTS_MODE',
 'APARTMENTS_AVG',
 'APARTMENTS_MEDI',
 'ELEVATORS_MODE',
 'ELEVATORS_MEDI',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'ENTRANCES_MODE',
 'ENTRANCES_MEDI',
 'AMT_ANNUITY',
 'REGION_POPULATION_RELATIVE',
 'AMT_GOODS_PRICE',
 'FLOORSMAX_MODE',
 'FLOORSMAX_MEDI',
 'AMT_CREDIT',
 'FLOORSMAX_AVG',
 'YEARS_BUILD_MODE']

I will leave Numerical Count variables as they are, even if they have high skewness, since they have a direct interpretation.

I do not apply log1p to time variables (days) because they may contain negative values, as they represent days relative to the application.

External Scores also do not undergo logarithmic transformation because they are already on a limited scale.

6.4 Scaling Strategy

For most of the transformations I need to perform, I will use StandardScaler:

$$
z=\frac{x-\mu}{\sigma}
$$

This transforms each feature so that mean=0 and std=1.

I do not use MinMaxScaler because it scales everything to a 0–1 range and is extremely sensitive to outliers, which works against me for variables like AMT_INCOME_TOTAL, where the maximum value is 117,000,000.

RobustScaler is the only alternative I will consider.

6.5 Features That Should Not Be Scaled

Not all of our features will be scaled using StandardScaler; the groups that are excluded are:

- Binary / Flag.

- Variables generated using one-hot encoding are already 0/1; there is no point in scaling them.

- Identifier: This has no relationship to the target and contributes nothing to the prediction; it will be excluded.

- Ordinal: Features classified as ordinal are already encoded as 1/2/3... and will remain unscaled so as not to lose their ordinal interpretation.

6.6 Model-Specific Scaling Strategy

Data scaling depends on the algorithm, so we will not create a single, permanently scaled dataset.

Logistic Regression --> Yes

KNN --> Yes

SVM --> Yes

Neural Network --> Yes

Random Forest --> No

Decision Tree --> No

Gradient Boosted Trees --> No

XGBoost --> No

LightGBM --> No


This leads me to have different pipelines depending on the algorithm

6.7 Transformation Decision Log

In [75]:
transformation_strategy = []

In [76]:
for feature in scaling_candidate_features:
    analytical_type = data_types.loc[data_types["Feature"] == feature, "Analytical Type"].iloc[0]
    skewness_row = skewness_summary.loc[skewness_summary["Feature"] == feature]

    if len(skewness_row) > 0:
        skewness = skewness_row["Skewness"].iloc[0]
        skewness_level = skewness_row["Skewness Level"].iloc[0]
    else:
        skewness = np.nan
        skewness_level = "Not Available"

    if analytical_type == "Numerical Continuous":
        if feature in log_transform_candidates:
            transformation = "Candidate for log1p transformation"
            reason = "Continuous non-negative feature with high positive skewness."
        else:
            transformation = "Keep original"
            reason = "No distribution transformation selected at this stage."
        scaling = "StandardScaler for scale-sensitive models"
    elif analytical_type == "Numerical Count":
        transformation = "Keep original"
        scaling = "StandardScaler for scale-sensitive models"
        reason = "Count feature retains direct interpretation; no automatic log transformation applied."
    elif analytical_type == "Temporal / Days":
        transformation = "Keep original"
        scaling = "StandardScaler for scale-sensitive models"
        reason = "Raw temporal values may be negative and are not suitable for automatic log1p transformation."
    elif analytical_type == "External Score":
        transformation = "Keep original"
        scaling = "StandardScaler for scale-sensitive models"
        reason = "External scores are already bounded numerical measures."

    transformation_strategy.append({
        "Feature": feature,
        "Analytical Type": analytical_type,
        "Skewness": skewness,
        "Skewness Level": skewness_level,
        "Distribution Transformation": transformation,
        "Scaling Strategy": scaling,
        "Applied Now": False,
        "Reason": reason
    })

In [77]:
transformation_strategy = pd.DataFrame(
    transformation_strategy
)

transformation_strategy

,Feature,Analytical Type,Skewness,Skewness Level,Distribution Transformation,Scaling Strategy,Applied Now,Reason
0,AMT_INCOME_TOTAL,Numerical Continuous,391.559654,High,Candidate for log1p transformation,StandardScaler for scale-sensitive models,False,Continuous non-negative feature with high posi...
1,AMT_CREDIT,Numerical Continuous,1.234778,High,Candidate for log1p transformation,StandardScaler for scale-sensitive models,False,Continuous non-negative feature with high posi...
2,AMT_ANNUITY,Numerical Continuous,1.579777,High,Candidate for log1p transformation,StandardScaler for scale-sensitive models,False,Continuous non-negative feature with high posi...
3,AMT_GOODS_PRICE,Numerical Continuous,1.349000,High,Candidate for log1p transformation,StandardScaler for scale-sensitive models,False,Continuous non-negative feature with high posi...
4,REGION_POPULATION_RELATIVE,Numerical Continuous,1.488009,High,Candidate for log1p transformation,StandardScaler for scale-sensitive models,False,Continuous non-negative feature with high posi...
...,...,...,...,...,...,...,...,...
65,OWN_CAR_AGE,Temporal / Days,2.745422,High,Keep original,StandardScaler for scale-sensitive models,False,Raw temporal values may be negative and are no...
66,DAYS_LAST_PHONE_CHANGE,Temporal / Days,-0.713606,Moderate,Keep original,StandardScaler for scale-sensitive models,False,Raw temporal values may be negative and are no...
67,EXT_SOURCE_1,External Score,-0.068755,Low,Keep original,StandardScaler for scale-sensitive models,False,External scores are already bounded numerical ...
68,EXT_SOURCE_2,External Score,-0.793576,Moderate,Keep original,StandardScaler for scale-sensitive models,False,External scores are already bounded numerical ...


In [78]:
transformation_strategy[transformation_strategy["Distribution Transformation"] == "Candidate for log1p transformation"][
    [
        "Feature",
        "Skewness",
        "Skewness Level",
        "Distribution Transformation"
    ]
]

,Feature,Skewness,Skewness Level,Distribution Transformation
0,AMT_INCOME_TOTAL,391.559654,High,Candidate for log1p transformation
1,AMT_CREDIT,1.234778,High,Candidate for log1p transformation
2,AMT_ANNUITY,1.579777,High,Candidate for log1p transformation
3,AMT_GOODS_PRICE,1.349000,High,Candidate for log1p transformation
4,REGION_POPULATION_RELATIVE,1.488009,High,Candidate for log1p transformation
6,APARTMENTS_AVG,2.641836,High,Candidate for log1p transformation
7,BASEMENTAREA_AVG,3.566306,High,Candidate for log1p transformation
8,YEARS_BEGINEXPLUATATION_AVG,-15.515264,High,Candidate for log1p transformation
10,COMMONAREA_AVG,5.457305,High,Candidate for log1p transformation
11,ELEVATORS_AVG,2.439429,High,Candidate for log1p transformation


In [79]:
transformation_strategy.to_csv("../outputs/data_preparation/transformation_strategy.csv",index=False)

## 7. Feature Engineering

7.1 Feature Engineering Strategy

I have features that don’t tell us anything on their own, such as AMT_CREDIT=500,000; however, Credit / Income = 5 does provide information indicating that the requested credit is equal to 5 times the reported annual income.

That’s why we’ve created four groups:

- Financial Ratios
- Temporal Features
- Household Features
- External / Behavioral Aggregates

I always retain the original columns; we only add new, valuable information.

7.2 Financial Ratio Features

In [80]:
#Function to prevent division by zero and pipeline breaks
def safe_divide(numerator, denominator):
    return np.where(
        denominator.notna() & (denominator != 0),
        numerator / denominator,
        np.nan
    )

Credit-to-Income Ratio

$$
\text{CREDIT\_INCOME\_RATIO}
=
\frac{\text{AMT\_CREDIT}}
{\text{AMT\_INCOME\_TOTAL}}
$$

It represents the size of the loan requested in relation to the applicant's income.

Conceptually, the higher this value is, the larger the loan will be relative to income.

In [81]:
#Credit-to-Income Ratio (credit_income_ratio = AMT_CREDIT / AMT_INCOME_TOTAL)
prepared_train_df["CREDIT_INCOME_RATIO"] = safe_divide(
    prepared_train_df["AMT_CREDIT"],
    prepared_train_df["AMT_INCOME_TOTAL"]
)

Annuity-to-Income Ratio

$$
\frac{\text{AMT\_ANNUITY}}
{\text{AMT\_INCOME\_TOTAL}}
$$

It represents the annuity payment relative to income. It allows us to see that even if two customers have the same annuity, their ability to pay may be completely different.

In [82]:
prepared_train_df["ANNUITY_INCOME_RATIO"] = safe_divide(
    prepared_train_df["AMT_ANNUITY"],
    prepared_train_df["AMT_INCOME_TOTAL"]
)

Credit-to-Annuity Ratio

$$
\frac{\text{AMT\_CREDIT}}
{\text{AMT\_ANNUITY}}
$$

This ratio allows us to estimate the relationship between credit and periodic payments.

In [83]:
prepared_train_df["CREDIT_ANNUITY_RATIO"] = safe_divide(
    prepared_train_df["AMT_CREDIT"],
    prepared_train_df["AMT_ANNUITY"]
)

Credit-to-Goods Ratio

$$
\frac{\text{AMT\_CREDIT}}
{\text{AMT\_GOODS\_PRICE}}
$$

It allows us to compare the loan amount with the value of the goods financed.

In [84]:
prepared_train_df["CREDIT_GOODS_RATIO"] = safe_divide(
    prepared_train_df["AMT_CREDIT"],
    prepared_train_df["AMT_GOODS_PRICE"]
)

7.3 Age and Temporal Features

Modify variables such as: DAYS_BIRTH, DAYS_EMPLOYED, DAYS_REGISTRATION, DAYS_ID_PUBLISH, DAYS_LAST_PHONE_CHANGE. It is much more intuitive to work with years than with the current values, such as -14,425 days.

In [85]:
prepared_train_df["AGE_YEARS"] = -prepared_train_df["DAYS_BIRTH"] / 365.25

In [86]:
prepared_train_df["EMPLOYMENT_YEARS"] = -prepared_train_df["DAYS_EMPLOYED"] / 365.25

In [87]:
prepared_train_df["REGISTRATION_YEARS"] = -prepared_train_df["DAYS_REGISTRATION"] / 365.25

In [88]:
prepared_train_df["ID_PUBLISH_YEARS"] = -prepared_train_df["DAYS_ID_PUBLISH"] / 365.25

In [89]:
prepared_train_df["PHONE_CHANGE_YEARS"] = -prepared_train_df["DAYS_LAST_PHONE_CHANGE"] / 365.25

I can also derive useful information such as the Employment-to-Age Ratio

$$
\frac{\text{Years Employed}}
{\text{Age}}
$$

This gives me an estimate of what proportion of the applicant's life has been associated with their current registered employment. I include a different ratio that can be used to assess stability for loan repayment.

In [90]:
prepared_train_df["EMPLOYMENT_AGE_RATIO"] = safe_divide(
    prepared_train_df["EMPLOYMENT_YEARS"],
    prepared_train_df["AGE_YEARS"]
)

7.4 Household Features

In [91]:
#Income per family member -> The same income does not carry the same weight in two families with different numbers of members
prepared_train_df["INCOME_PER_PERSON"] = safe_divide(
    prepared_train_df["AMT_INCOME_TOTAL"],
    prepared_train_df["CNT_FAM_MEMBERS"]
)

In [92]:
#Children Ratio
prepared_train_df["CHILDREN_RATIO"] = safe_divide(
    prepared_train_df["CNT_CHILDREN"],
    prepared_train_df["CNT_FAM_MEMBERS"]
)

7.5 External Score Features

In the 01_data_understanding notebook, I show the strong correlation between TARGET and the features: EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3. Therefore, for each customer, I create a new value that is the average of the three values assigned to that customer.

In [93]:
#Calculate External Score Mean
external_sources = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]
prepared_train_df["EXT_SOURCE_MEAN"] = prepared_train_df[external_sources].mean(axis=1)

In [94]:
#Calculate External Score Minimum
prepared_train_df["EXT_SOURCE_MIN"] = prepared_train_df[external_sources].min(axis=1)

In [95]:
#Calculate External Score Maximum
prepared_train_df["EXT_SOURCE_MAX"] = prepared_train_df[external_sources].max(axis=1)

In [96]:
#Calculate External Score Standard Deviation
prepared_train_df["EXT_SOURCE_STD"] = prepared_train_df[external_sources].std(axis=1)

In [97]:
#External Scores available
prepared_train_df["EXT_SOURCE_COUNT"] = prepared_train_df[external_sources].notna().sum(axis=1)

7.6 Document and Contact Features

Here, I reduce several FLAG variables from the data frame that contain relationships.

Document Count Features: FLAG_DOCUMENT_2, FLAG_DOCUMENT_3, etc...

Contact Information Count: FLAG_MOBIL, FLAG_WORK_PHONE, etc...

In [98]:
document_features = [column for column in prepared_train_df.columns if column.startswith("FLAG_DOCUMENT_")]
print("Document flag features:", len(document_features))
document_features

Document flag features: 20


['FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_3',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_5',
 'FLAG_DOCUMENT_6',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_8',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_10',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_15',
 'FLAG_DOCUMENT_16',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_21']

In [99]:
prepared_train_df["DOCUMENT_COUNT"] = prepared_train_df[document_features].sum(axis=1)

In [100]:
contact_features = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL"
]

[feature for feature in contact_features if feature not in prepared_train_df.columns]

[]

In [101]:
prepared_train_df["CONTACT_COUNT"] = prepared_train_df[contact_features].sum(axis=1)

7.7 Invalid Value Handling

In [102]:
engineered_features = [
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_ANNUITY_RATIO",
    "CREDIT_GOODS_RATIO",
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "PHONE_CHANGE_YEARS",
    "EMPLOYMENT_AGE_RATIO",
    "INCOME_PER_PERSON",
    "CHILDREN_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_COUNT",
    "DOCUMENT_COUNT",
    "CONTACT_COUNT"
]

In [103]:
infinite_summary = []

for feature in engineered_features:

    infinite_count = np.isinf(
        prepared_train_df[feature]
    ).sum()

    infinite_summary.append({
        "Feature": feature,
        "Infinite Values": infinite_count
    })

infinite_summary = pd.DataFrame(
    infinite_summary
)

infinite_summary

,Feature,Infinite Values
0,CREDIT_INCOME_RATIO,0
1,ANNUITY_INCOME_RATIO,0
2,CREDIT_ANNUITY_RATIO,0
3,CREDIT_GOODS_RATIO,0
4,AGE_YEARS,0
5,EMPLOYMENT_YEARS,0
6,REGISTRATION_YEARS,0
7,ID_PUBLISH_YEARS,0
8,PHONE_CHANGE_YEARS,0
9,EMPLOYMENT_AGE_RATIO,0


In [104]:
#Check how many NaN values are contained in the new features I've created
engineered_missing_summary = pd.DataFrame({
    "Feature": engineered_features,
    "Missing Count": [
        prepared_train_df[feature]
        .isna()
        .sum()
        for feature in engineered_features
    ]
})

In [105]:
#I add the percentages and sort the results.
engineered_missing_summary["Missing Percentage"] = engineered_missing_summary["Missing Count"] / len(prepared_train_df)* 100
engineered_missing_summary.sort_values("Missing Percentage",ascending=False)

,Feature,Missing Count,Missing Percentage
9,EMPLOYMENT_AGE_RATIO,55374,18.007161
5,EMPLOYMENT_YEARS,55374,18.007161
15,EXT_SOURCE_STD,36909,12.002497
3,CREDIT_GOODS_RATIO,278,0.090403
14,EXT_SOURCE_MAX,172,0.055933
12,EXT_SOURCE_MEAN,172,0.055933
13,EXT_SOURCE_MIN,172,0.055933
2,CREDIT_ANNUITY_RATIO,12,0.003902
1,ANNUITY_INCOME_RATIO,12,0.003902
10,INCOME_PER_PERSON,2,0.000650


In [106]:
#Check logical relationships
print("Negative AGE_YEARS:", (prepared_train_df["AGE_YEARS"] < 0).sum())
print("Negative EMPLOYMENT_YEARS:",(prepared_train_df["EMPLOYMENT_YEARS"] < 0).sum())
print("Infinite engineered values:", np.isinf(prepared_train_df[engineered_features]).sum().sum())

Negative AGE_YEARS: 0
Negative EMPLOYMENT_YEARS: 0
Infinite engineered values: 0


7.8 Engineered Feature Metadata

When creating new features, I must update the data_types.csv file, which is our master data source where each feature is classified by its function; I must update it with the new ones.

In [107]:
engineered_metadata = pd.DataFrame({
    "Feature": [
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "CREDIT_ANNUITY_RATIO",
        "CREDIT_GOODS_RATIO",
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "REGISTRATION_YEARS",
        "ID_PUBLISH_YEARS",
        "PHONE_CHANGE_YEARS",
        "EMPLOYMENT_AGE_RATIO",
        "INCOME_PER_PERSON",
        "CHILDREN_RATIO",
        "EXT_SOURCE_MEAN",
        "EXT_SOURCE_MIN",
        "EXT_SOURCE_MAX",
        "EXT_SOURCE_STD",
        "EXT_SOURCE_COUNT",
        "DOCUMENT_COUNT",
        "CONTACT_COUNT"
    ],

    "Analytical Type": [
        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",

        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",

        "Numerical Continuous",
        "Numerical Continuous",
        "Numerical Continuous",

        "External Score",
        "External Score",
        "External Score",
        "Numerical Continuous",

        "Numerical Count",
        "Numerical Count",
        "Numerical Count"
    ]
})

In [108]:
engineered_metadata["Pandas Type"] = [str(prepared_train_df[feature].dtype) for feature in engineered_metadata["Feature"]]

In [109]:
engineered_metadata = engineered_metadata[
    [
        "Feature",
        "Pandas Type",
        "Analytical Type"
    ]
]

In [110]:
#Update metadata
existing_engineered_features = (set(engineered_metadata["Feature"]) & set(data_types["Feature"]))
existing_engineered_features

set()

In [111]:
data_types = pd.concat([data_types, engineered_metadata], ignore_index=True)

In [112]:
data_types.tail(len(engineered_metadata))

,Unnamed: 0,Feature,Pandas Type,Analytical Type
122,NaN,CREDIT_INCOME_RATIO,float64,Numerical Continuous
123,NaN,ANNUITY_INCOME_RATIO,float64,Numerical Continuous
124,NaN,CREDIT_ANNUITY_RATIO,float64,Numerical Continuous
125,NaN,CREDIT_GOODS_RATIO,float64,Numerical Continuous
126,NaN,AGE_YEARS,float64,Numerical Continuous
127,NaN,EMPLOYMENT_YEARS,float64,Numerical Continuous
128,NaN,REGISTRATION_YEARS,float64,Numerical Continuous
129,NaN,ID_PUBLISH_YEARS,float64,Numerical Continuous
130,NaN,PHONE_CHANGE_YEARS,float64,Numerical Continuous
131,NaN,EMPLOYMENT_AGE_RATIO,float64,Numerical Continuous


In [113]:
#Save metadata
data_types.to_csv("../outputs/data_preparation/data_types_prepared.csv", index=False)

7.9 Verification

In [114]:
#Number of features before and after
features_before_engineering = prepared_train_df.shape[1] - len(engineered_features)

In [115]:
features_after_engineering = prepared_train_df.shape[1]

print("Features before engineering:", features_before_engineering)
print("Features after engineering:", features_after_engineering)
print("New engineered features:", len(engineered_features))

Features before engineering: 123
Features after engineering: 142
New engineered features: 19


In [116]:
assert (features_after_engineering == features_before_engineering + len(engineered_features))

In [117]:
missing_engineered_columns = [feature for feature in engineered_features if feature not in prepared_train_df.columns]
missing_engineered_columns

[]

In [118]:
assert len(missing_engineered_columns) == 0

In [119]:
#Check that I haven't deleted any rows from the original dataframe
print("Original rows:", len(train_df))
print("Prepared rows:", len(prepared_train_df))

Original rows: 307511
Prepared rows: 307511


-----------------------------------------------------------------------------------------------------------------------------------

**PREVIEW ENHANCED DATAFRAME**

In [120]:
prepared_train_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,EMPLOYMENT_AGE_RATIO,INCOME_PER_PERSON,CHILDREN_RATIO,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_COUNT,DOCUMENT_COUNT,CONTACT_COUNT
0,100002,1,Cash loans,M,0,1,0,202500.0,406597.5,24700.5,...,0.067329,202500.0,0.0,0.161787,0.083037,0.262949,0.092026,3,1,4
1,100003,0,Cash loans,F,0,0,0,270000.0,1293502.5,35698.5,...,0.070862,135000.0,0.0,0.466757,0.311267,0.622246,0.219895,2,1,4
2,100004,0,Revolving loans,M,1,1,0,67500.0,135000.0,6750.0,...,0.011814,67500.0,0.0,0.642739,0.555912,0.729567,0.122792,2,0,5
3,100006,0,Cash loans,F,0,1,0,135000.0,312682.5,29686.5,...,0.159905,67500.0,0.0,0.650442,0.650442,0.650442,NaN,1,1,3
4,100007,0,Cash loans,M,0,1,0,121500.0,513000.0,21865.5,...,0.152418,121500.0,0.0,0.322738,0.322738,0.322738,NaN,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,0,0,0,157500.0,254700.0,27558.0,...,0.025303,157500.0,0.0,0.413601,0.145570,0.681632,0.379053,2,1,3
307507,456252,0,Cash loans,F,0,1,0,72000.0,269550.0,12001.5,...,NaN,72000.0,0.0,0.115992,0.115992,0.115992,NaN,1,1,3
307508,456253,0,Cash loans,F,0,1,0,153000.0,677664.0,29979.0,...,0.529266,153000.0,0.0,0.499536,0.218859,0.744026,0.264447,3,1,4
307509,456254,1,Cash loans,F,0,1,0,171000.0,370107.0,20205.0,...,0.400134,85500.0,0.0,0.587593,0.514163,0.661024,0.103846,2,1,3


-----------------------------------------------------------------------------------------------------------------------------------

## 8. FEATURE SELECTION

In [121]:
#Save initial state
features_before_selection = prepared_train_df.shape[1]
print("Features before selection:",features_before_selection)

Features before selection: 142


In [122]:
features_to_remove = []
feature_selection_log = []

8.2 Identifier and Non-Predictive Features

The SK_ID_CURR column is our unique identifier. It will not be used in the prediction process, but it is necessary for joining auxiliary tables and subsequent feature engineering processes.

In [123]:
identifier_features = data_types.loc[data_types["Analytical Type"] == "Identifier", "Feature"].tolist()

identifier_features

['SK_ID_CURR']

In [124]:
for feature in identifier_features:

    feature_selection_log.append({
        "Feature": feature,
        "Selection Category": "Identifier",
        "Decision": "Exclude from predictors",
        "Applied Now": False,
        "Reason": "Identifier retained for joins and traceability but excluded from model predictors."
    })

In [125]:
target_feature = "TARGET"

8.3 Redundant Engineered Features

In [126]:
temporal_redundancy_pairs = {
    "DAYS_BIRTH": "AGE_YEARS",
    "DAYS_EMPLOYED": "EMPLOYMENT_YEARS",
    "DAYS_REGISTRATION": "REGISTRATION_YEARS",
    "DAYS_ID_PUBLISH": "ID_PUBLISH_YEARS",
    "DAYS_LAST_PHONE_CHANGE": "PHONE_CHANGE_YEARS"
}

Keep engineered representations & Drop raw DAYS representation from predictors

In [127]:
#I removed the 5 features previously identified as redundant from `prepared_train_df`. Previously 142 columns, now 137 columns.
redundant_temporal_features = list(temporal_redundancy_pairs.keys())
prepared_train_df = prepared_train_df.drop(columns=redundant_temporal_features)

In [128]:
#Definir en la lista de features eliminadas las borradas anteriormente.
for raw_feature, engineered_feature in temporal_redundancy_pairs.items():
    features_to_remove.append(raw_feature)
    feature_selection_log.append({
        "Feature": raw_feature,
        "Selection Category": "Deterministic Redundancy",
        "Decision": "Remove",
        "Applied Now": True,
        "Reason": (
            f"Exact mathematical information is retained "
            f"in the more interpretable feature "
            f"{engineered_feature}."
        )
    })

In [129]:
prepared_train_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,EMPLOYMENT_AGE_RATIO,INCOME_PER_PERSON,CHILDREN_RATIO,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_COUNT,DOCUMENT_COUNT,CONTACT_COUNT
0,100002,1,Cash loans,M,0,1,0,202500.0,406597.5,24700.5,...,0.067329,202500.0,0.0,0.161787,0.083037,0.262949,0.092026,3,1,4
1,100003,0,Cash loans,F,0,0,0,270000.0,1293502.5,35698.5,...,0.070862,135000.0,0.0,0.466757,0.311267,0.622246,0.219895,2,1,4
2,100004,0,Revolving loans,M,1,1,0,67500.0,135000.0,6750.0,...,0.011814,67500.0,0.0,0.642739,0.555912,0.729567,0.122792,2,0,5
3,100006,0,Cash loans,F,0,1,0,135000.0,312682.5,29686.5,...,0.159905,67500.0,0.0,0.650442,0.650442,0.650442,NaN,1,1,3
4,100007,0,Cash loans,M,0,1,0,121500.0,513000.0,21865.5,...,0.152418,121500.0,0.0,0.322738,0.322738,0.322738,NaN,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,0,0,0,157500.0,254700.0,27558.0,...,0.025303,157500.0,0.0,0.413601,0.145570,0.681632,0.379053,2,1,3
307507,456252,0,Cash loans,F,0,1,0,72000.0,269550.0,12001.5,...,NaN,72000.0,0.0,0.115992,0.115992,0.115992,NaN,1,1,3
307508,456253,0,Cash loans,F,0,1,0,153000.0,677664.0,29979.0,...,0.529266,153000.0,0.0,0.499536,0.218859,0.744026,0.264447,3,1,4
307509,456254,1,Cash loans,F,0,1,0,171000.0,370107.0,20205.0,...,0.400134,85500.0,0.0,0.587593,0.514163,0.661024,0.103846,2,1,3


8.4 Constant Features

I have columns that are purely constant; they have the same value in all their records.

At this point, I decided not to run ‘prepared_train_df.nunique()’ because this would cause data leakage—since I would be removing data based on the result obtained that would later appear in the validation or test sets and does not all belong to the training set.

After I split the final DataFrame, the technique to use will be:

nunique(dropna=False) <= 1
→ Remove

In [130]:
constant_candidates = [feature for feature in prepared_train_df.columns if feature not in ["TARGET", "SK_ID_CURR"] and prepared_train_df[feature].nunique(dropna=False) <= 1]

constant_candidates

[]

In [131]:
constant_candidates_df = pd.DataFrame({
    "Feature": constant_candidates,
    "Preliminary Status": "Constant candidate",
    "Final Decision": (
        "Re-evaluate using internal training set"
    )
})

8.5 Near-Constant Features

“Near-Constant Features” refers to variables that contain virtually the same values without actually being constants. Technically, the column changes, but it contains very little variability; this does not mean we should immediately remove it, but we will review these variables.

In [132]:
#Calculate the mean for each feature to determine the most frequent value
near_constant_summary = []
candidate_predictors = [feature for feature in prepared_train_df.columns if feature not in ["TARGET", "SK_ID_CURR"]]

In [133]:
for feature in candidate_predictors:
    value_counts = prepared_train_df[feature].value_counts(normalize=True, dropna=False)

    if len(value_counts) > 0:
        dominant_percentage = value_counts.iloc[0] * 100
    else:
        dominant_percentage = np.nan

    near_constant_summary.append({
        "Feature": feature,
        "Dominant Value Percentage": dominant_percentage
    })

In [134]:
near_constant_summary = pd.DataFrame(near_constant_summary)
near_constant_summary = near_constant_summary.sort_values("Dominant Value Percentage", ascending=False)
near_constant_summary.head(20)

,Feature,Dominant Value Percentage
16,FLAG_MOBIL,99.999675
99,FLAG_DOCUMENT_12,99.999350
97,FLAG_DOCUMENT_10,99.997724
89,FLAG_DOCUMENT_2,99.995773
91,FLAG_DOCUMENT_4,99.991870
94,FLAG_DOCUMENT_7,99.980814
104,FLAG_DOCUMENT_17,99.973334
108,FLAG_DOCUMENT_21,99.966505
107,FLAG_DOCUMENT_20,99.949270
106,FLAG_DOCUMENT_19,99.940490


Para decidir si eliminar la feature usamos un threshold igual que hicimos en screening anteriormente '>= 99%'. Pero no podemos hacer eliminación directa ya que tendriamos data leakage igual que anteriormente y lo realizamos después de la división del dataframe.

In [135]:
#Displaying the features to be removed after applying the threshold
near_constant_candidates = near_constant_summary[near_constant_summary["Dominant Value Percentage"] >= 99]

near_constant_candidates

,Feature,Dominant Value Percentage
16,FLAG_MOBIL,99.999675
99,FLAG_DOCUMENT_12,99.999350
97,FLAG_DOCUMENT_10,99.997724
89,FLAG_DOCUMENT_2,99.995773
91,FLAG_DOCUMENT_4,99.991870
94,FLAG_DOCUMENT_7,99.980814
104,FLAG_DOCUMENT_17,99.973334
108,FLAG_DOCUMENT_21,99.966505
107,FLAG_DOCUMENT_20,99.949270
106,FLAG_DOCUMENT_19,99.940490


8.6 Highly Correlated Features

To identify features that are highly correlated and remove them—since they provide the same information to the model I use the following rule:

|Pearson correlation| >= 0.98

This allows me to identify non-duplicate numerical features; by setting such a high threshold, I ensure that the features I remove are practically redundant.

The correlation involves two features, and I cannot eliminate both even if their correlation exceeds 98%; therefore, the criteria I will follow are:

1. Fewer Missing Values: Of the two features, I will choose the one with fewer missing values, since it contains more actual observations.

2. More Interpretable: If both have a similar number of missing values, I will choose the one that is more interpretable for the TARGET.

8.7 Missingness-Based Feature Review

Previously, I classified the number of missing values in a feature as LOW, MEDIUM, and HIGH, but this does not mean I should remove it outright. For that to happen, the following conditions must be met:

High missingness
+
high redundancy
+
weak interpretability
+
an alternative feature contains the same information

In [136]:
#Analyze missing values per feature
missing_strategy = pd.read_csv("../outputs/data_preparation/missing_values_strategy.csv")

high_missing_candidates = missing_strategy[missing_strategy["Rate"] == "HIGH"]
high_missing_candidates

,Unnamed: 0,Feature,Analytical Type,Selected,Reason,Missing %,Rate,Method Apply


8.8 Final Feature Selection Rules

Identifier -> Exclude from predictors

Target -> Separate from predictors

Exact mathematical redundancy -> Remove duplicate representation

Constant feature -> Remove based on internal train

Near-constant ≥99% -> Review on internal train

Correlation ≥0.98 -> Review redundancy on internal train

High missingness only -> Keep / review

High missingness + strong redundancy -> Candidate for removal

Engineered feature -> Keep unless redundant

Unique business information -> Prefer keep

8.9 Feature Selection Log

In [137]:
feature_selection_log = pd.DataFrame(feature_selection_log)

In [138]:
constant_log = pd.DataFrame({
    "Feature": constant_candidates,
    "Selection Category": "Constant Candidate",
    "Decision": "Re-evaluate on internal train",
    "Applied Now": False,
    "Reason": "Feature showed no variability in preliminary screening."
})

In [139]:
near_constant_log = pd.DataFrame({
    "Feature": near_constant_candidates["Feature"],
    "Selection Category": "Near-Constant Candidate",
    "Decision": "Review on internal train",
    "Applied Now": False,
    "Reason": "At least 99% of preliminary observations share the same value."
})

In [140]:
feature_selection_log = pd.concat(
    [
        feature_selection_log,
        constant_log,
        near_constant_log
    ],
    ignore_index=True
)

In [141]:
feature_selection_log.to_csv("../outputs/data_preparation/feature_selection_log.csv", index=False)
near_constant_summary.to_csv("../outputs/data_preparation/near_constant_summary.csv", index=False)
constant_candidates_df.to_csv("../outputs/data_preparation/constant_candidates.csv", index=False)

CURRENT LIST OF FEATURES FOR PREDICTION

In [142]:
current_predictor_features = [feature for feature in prepared_train_df.columns if feature not in ["SK_ID_CURR","TARGET"]]

In [143]:
print("Current predictor features:", len(current_predictor_features))
current_predictor_features

Current predictor features: 135


['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'F

In [144]:
features_after_selection = (prepared_train_df.shape[1])

print("Features before Section 8:", features_before_selection)
print("Features after deterministic selection:",features_after_selection)
print("Features removed now:",len(features_to_remove))

Features before Section 8: 142
Features after deterministic selection: 137
Features removed now: 5


In [145]:
data_types = data_types[~data_types["Feature"].isin(redundant_temporal_features)].copy()
data_types[data_types["Feature"].isin(redundant_temporal_features)]
data_types.to_csv("../outputs/data_preparation/data_types_prepared.csv",index=False)

In [146]:
df_features = set(prepared_train_df.columns)
metadata_features = set(data_types["Feature"])

missing_in_metadata = sorted(df_features - metadata_features)
missing_in_dataframe = sorted(metadata_features - df_features)

print("Present in dataframe but missing in metadata:", missing_in_metadata)
print("Present in metadata but missing in dataframe:", missing_in_dataframe)

Present in dataframe but missing in metadata: ['DAYS_EMPLOYED_ANOM']
Present in metadata but missing in dataframe: []


In [147]:
days_employed_anom_metadata = pd.DataFrame({
    "Feature": ["DAYS_EMPLOYED_ANOM"],
    "Pandas Type": [str(prepared_train_df["DAYS_EMPLOYED_ANOM"].dtype)],
    "Analytical Type": ["Binary / Flag"]
})

data_types = pd.concat([data_types, days_employed_anom_metadata], ignore_index=True)

In [148]:
data_types.to_csv("../outputs/data_preparation/data_types_prepared.csv", index=False)

In [149]:
prepared_train_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,EMPLOYMENT_AGE_RATIO,INCOME_PER_PERSON,CHILDREN_RATIO,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_COUNT,DOCUMENT_COUNT,CONTACT_COUNT
0,100002,1,Cash loans,M,0,1,0,202500.0,406597.5,24700.5,...,0.067329,202500.0,0.0,0.161787,0.083037,0.262949,0.092026,3,1,4
1,100003,0,Cash loans,F,0,0,0,270000.0,1293502.5,35698.5,...,0.070862,135000.0,0.0,0.466757,0.311267,0.622246,0.219895,2,1,4
2,100004,0,Revolving loans,M,1,1,0,67500.0,135000.0,6750.0,...,0.011814,67500.0,0.0,0.642739,0.555912,0.729567,0.122792,2,0,5
3,100006,0,Cash loans,F,0,1,0,135000.0,312682.5,29686.5,...,0.159905,67500.0,0.0,0.650442,0.650442,0.650442,NaN,1,1,3
4,100007,0,Cash loans,M,0,1,0,121500.0,513000.0,21865.5,...,0.152418,121500.0,0.0,0.322738,0.322738,0.322738,NaN,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,0,0,0,157500.0,254700.0,27558.0,...,0.025303,157500.0,0.0,0.413601,0.145570,0.681632,0.379053,2,1,3
307507,456252,0,Cash loans,F,0,1,0,72000.0,269550.0,12001.5,...,NaN,72000.0,0.0,0.115992,0.115992,0.115992,NaN,1,1,3
307508,456253,0,Cash loans,F,0,1,0,153000.0,677664.0,29979.0,...,0.529266,153000.0,0.0,0.499536,0.218859,0.744026,0.264447,3,1,4
307509,456254,1,Cash loans,F,0,1,0,171000.0,370107.0,20205.0,...,0.400134,85500.0,0.0,0.587593,0.514163,0.661024,0.103846,2,1,3


## 9. Auxiliary Table Integration

So far, I have worked only with ‘application_train.csv,’ but the dataset also includes information in auxiliary tables such as: bureau.csv, bureau_balance.csv, previous_application.csv, installments_payments.csv, POS_CASH_balance.csv, and credit_card_balance.csv; these tables contain potentially valuable historical information about the applicant's past credit behavior.

Since my goal is to have two distinct dataframes—‘prepared_train_df’, which is the baseline, and ‘advanced_train_df’, which will already include these features from the auxiliary tables.

9.1 Integration Strategy

In [150]:
advanced_train_df = prepared_train_df.copy()

In [151]:
print("Baseline dataset:", prepared_train_df.shape)
print("Advanced dataset before integration:", advanced_train_df.shape)

Baseline dataset: (307511, 137)
Advanced dataset before integration: (307511, 137)


From this point on, ‘prepared_train_df’ will serve as the baseline and will remain unchanged, while ‘advanced_train_df’ will be the one that receives the information from these auxiliary tables.

9.2 Load Auxiliary Tables

In [152]:
bureau_df = pd.read_csv("../data/bureau.csv")
bureau_balance_df = pd.read_csv("../data/bureau_balance.csv")
previous_application_df = pd.read_csv( "../data/previous_application.csv")
installments_df = pd.read_csv("../data/installments_payments.csv")
pos_cash_df = pd.read_csv("../data/POS_CASH_balance.csv")
credit_card_df = pd.read_csv("../data/credit_card_balance.csv")

In [153]:
#Create an inventory of auxiliary tables and save it
auxiliary_inventory = pd.DataFrame({
    "Table": [
        "bureau",
        "bureau_balance",
        "previous_application",
        "installments_payments",
        "POS_CASH_balance",
        "credit_card_balance"
    ],
    "Rows": [
        len(bureau_df),
        len(bureau_balance_df),
        len(previous_application_df),
        len(installments_df),
        len(pos_cash_df),
        len(credit_card_df)
    ],
    "Columns": [
        bureau_df.shape[1],
        bureau_balance_df.shape[1],
        previous_application_df.shape[1],
        installments_df.shape[1],
        pos_cash_df.shape[1],
        credit_card_df.shape[1]
    ]
})

auxiliary_inventory

,Table,Rows,Columns
0,bureau,1716428,17
1,bureau_balance,27299925,3
2,previous_application,1670214,37
3,installments_payments,13605401,8
4,POS_CASH_balance,10001358,8
5,credit_card_balance,3840312,23


In [154]:
auxiliary_inventory.to_csv("../outputs/data_preparation/auxiliary_table_inventory.csv", index=False)

9.3 Bureau Balance Aggregation

In [155]:
bureau_balance_df["BB_OVERDUE_FLAG"] = bureau_balance_df["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)
bureau_balance_df["BB_SEVERE_OVERDUE_FLAG"] = bureau_balance_df["STATUS"].isin(["3", "4", "5"]).astype(int)

In [156]:
#Aggregate per SK_ID_BUREAU
bureau_balance_agg = (
    bureau_balance_df
    .groupby("SK_ID_BUREAU")
    .agg(
        BB_MONTH_COUNT=("MONTHS_BALANCE","count"),
        BB_MONTHS_BALANCE_MIN=("MONTHS_BALANCE","min"),
        BB_MONTHS_BALANCE_MAX=("MONTHS_BALANCE","max"),
        BB_OVERDUE_COUNT=("BB_OVERDUE_FLAG","sum"),
        BB_SEVERE_OVERDUE_COUNT=("BB_SEVERE_OVERDUE_FLAG","sum")
    )
    .reset_index()
)

In [157]:
#Create ratios
bureau_balance_agg["BB_OVERDUE_RATE"] = safe_divide(bureau_balance_agg["BB_OVERDUE_COUNT"], bureau_balance_agg["BB_MONTH_COUNT"])
bureau_balance_agg["BB_SEVERE_OVERDUE_RATE"] = safe_divide(bureau_balance_agg["BB_SEVERE_OVERDUE_COUNT"], bureau_balance_agg["BB_MONTH_COUNT"])

9.4 Bureau Aggregation

In [158]:
#I've included the summary created above in the DF
bureau_enriched_df = bureau_df.merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left",
    validate="one_to_one"
)

In [159]:
bureau_enriched_df["BUREAU_ACTIVE_FLAG"] = (bureau_enriched_df["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_enriched_df["BUREAU_CLOSED_FLAG"] = (bureau_enriched_df["CREDIT_ACTIVE"] == "Closed").astype(int)

In [160]:
#Aggregate it for each client
bureau_customer_agg = (
    bureau_enriched_df
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_CREDIT_COUNT=("SK_ID_BUREAU","count"),
        BUREAU_ACTIVE_COUNT=("BUREAU_ACTIVE_FLAG","sum"),
        BUREAU_CLOSED_COUNT=("BUREAU_CLOSED_FLAG","sum"),
        BUREAU_CREDIT_SUM_TOTAL=("AMT_CREDIT_SUM","sum"),
        BUREAU_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM","mean"),
        BUREAU_CREDIT_SUM_MAX=("AMT_CREDIT_SUM","max"),
        BUREAU_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT","sum"),
        BUREAU_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT","mean"),
        BUREAU_OVERDUE_TOTAL=("AMT_CREDIT_SUM_OVERDUE","sum"),
        BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE","max"),
        BUREAU_CREDIT_PROLONG_TOTAL=("CNT_CREDIT_PROLONG","sum"),
        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT","mean"),
        BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT","min"),
        BUREAU_OVERDUE_MONTHS_TOTAL=("BB_OVERDUE_COUNT","sum"),
        BUREAU_SEVERE_OVERDUE_MONTHS_TOTAL=("BB_SEVERE_OVERDUE_COUNT","sum")
    )
    .reset_index()
)

In [161]:
#We create an interpretable ratio and an active loan ratio -> This gives us one row for each SK_ID_CURR
bureau_customer_agg["BUREAU_DEBT_CREDIT_RATIO"] = safe_divide(bureau_customer_agg["BUREAU_DEBT_TOTAL"], bureau_customer_agg["BUREAU_CREDIT_SUM_TOTAL"])
bureau_customer_agg["BUREAU_ACTIVE_RATE"] = safe_divide(bureau_customer_agg["BUREAU_ACTIVE_COUNT"], bureau_customer_agg["BUREAU_CREDIT_COUNT"])

9.5 Previous Applications Aggregation

This table contains the customer's previous applications with Home Credit.

In [162]:
#Create status flags
previous_application_df["PREV_APPROVED_FLAG"] = (previous_application_df["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
previous_application_df["PREV_REFUSED_FLAG"] = (previous_application_df["NAME_CONTRACT_STATUS"] == "Refused").astype(int)

In [163]:
#Aggregate
previous_customer_agg = (
    previous_application_df
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=("SK_ID_PREV","count"),
        PREV_APPROVED_COUNT=("PREV_APPROVED_FLAG","sum"),
        PREV_REFUSED_COUNT=("PREV_REFUSED_FLAG","sum"),
        PREV_APPLICATION_AMOUNT_MEAN=("AMT_APPLICATION","mean"),
        PREV_APPLICATION_AMOUNT_MAX=("AMT_APPLICATION","max"),
        PREV_CREDIT_AMOUNT_MEAN=("AMT_CREDIT","mean"),
        PREV_CREDIT_AMOUNT_MAX=("AMT_CREDIT","max"),
        PREV_ANNUITY_MEAN=("AMT_ANNUITY","mean"),
        PREV_DAYS_DECISION_MEAN=("DAYS_DECISION","mean"),
        PREV_DAYS_DECISION_MIN=("DAYS_DECISION","min"),
        PREV_PAYMENT_COUNT_MEAN=("CNT_PAYMENT","mean")
    )
    .reset_index()
)

In [164]:
#I track each applicant's history with Home Credit
previous_customer_agg["PREV_APPROVAL_RATE"] = safe_divide(previous_customer_agg["PREV_APPROVED_COUNT"], previous_customer_agg["PREV_APPLICATION_COUNT"])
previous_customer_agg["PREV_REFUSAL_RATE"] = safe_divide(previous_customer_agg["PREV_REFUSED_COUNT"], previous_customer_agg["PREV_APPLICATION_COUNT"])

9.6 Installments Payments Aggregation

In [165]:
installments_df[ "PAYMENT_DIFFERENCE"] = installments_df["AMT_INSTALMENT"] - installments_df["AMT_PAYMENT"]

Interpreting the payment differential allows us to express the difference between the scheduled payment and the actual payment made as a numerical value:

> 0 → paid less than scheduled

0 → paid the scheduled amount

< 0 → paid more than scheduled

In [166]:
#I Calculate Payment Delays
installments_df["PAYMENT_DELAY"] = installments_df["DAYS_ENTRY_PAYMENT"] - installments_df["DAYS_INSTALMENT"]

In [167]:
#I created the flag for late payments
installments_df["LATE_PAYMENT_FLAG"] = (installments_df["PAYMENT_DELAY"] > 0).astype(int)

In [168]:
#Aggregation
installments_customer_agg = (
    installments_df
    .groupby("SK_ID_CURR")
    .agg(
        INSTALMENT_RECORD_COUNT=("SK_ID_PREV","count"),
        INSTALMENT_AMOUNT_TOTAL=("AMT_INSTALMENT","sum"),
        PAYMENT_AMOUNT_TOTAL=("AMT_PAYMENT","sum"),
        PAYMENT_DIFFERENCE_MEAN=("PAYMENT_DIFFERENCE","mean"),
        PAYMENT_DIFFERENCE_MAX=("PAYMENT_DIFFERENCE","max"),
        PAYMENT_DELAY_MEAN=("PAYMENT_DELAY","mean"),
        PAYMENT_DELAY_MAX=("PAYMENT_DELAY","max"),
        LATE_PAYMENT_COUNT=("LATE_PAYMENT_FLAG","sum")
    )
    .reset_index()
)

In [169]:
installments_customer_agg["LATE_PAYMENT_RATE"] = safe_divide(installments_customer_agg["LATE_PAYMENT_COUNT"], installments_customer_agg["INSTALMENT_RECORD_COUNT"])

The ‘LATE_PAYMENT_RATE’ feature is highly likely to be important for the model, since it summarizes the customer's actual credit behavior.

In [170]:
installments_customer_agg["LATE_PAYMENT_RATE"]

0         0.142857
1         0.000000
2         0.000000
3         0.000000
4         0.111111
            ...   
339582    0.000000
339583    0.166667
339584    0.071429
339585    0.000000
339586    0.189189
Name: LATE_PAYMENT_RATE, Length: 339587, dtype: float64

9.7 POS Cash Balance Aggregation

In [171]:
#Create indicator and aggregate
pos_cash_df["POS_DPD_FLAG"] = (pos_cash_df["SK_DPD"] > 0).astype(int)

pos_customer_agg = (
    pos_cash_df
    .groupby("SK_ID_CURR")
    .agg(
        POS_RECORD_COUNT=("SK_ID_PREV","count"),
        POS_MONTHS_BALANCE_MIN=("MONTHS_BALANCE","min"),
        POS_INSTALMENT_MEAN=("CNT_INSTALMENT","mean"),
        POS_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE","mean"),
        POS_DPD_MEAN=("SK_DPD","mean"),
        POS_DPD_MAX=("SK_DPD","max"),
        POS_DPD_DEF_MEAN=("SK_DPD_DEF","mean"),
        POS_DPD_DEF_MAX=("SK_DPD_DEF","max"),
        POS_DPD_COUNT=("POS_DPD_FLAG","sum")
    )
    .reset_index()
)

pos_customer_agg["POS_DPD_RATE"] = safe_divide(pos_customer_agg["POS_DPD_COUNT"], pos_customer_agg["POS_RECORD_COUNT"])
pos_customer_agg

,SK_ID_CURR,POS_RECORD_COUNT,POS_MONTHS_BALANCE_MIN,POS_INSTALMENT_MEAN,POS_INSTALMENT_FUTURE_MEAN,POS_DPD_MEAN,POS_DPD_MAX,POS_DPD_DEF_MEAN,POS_DPD_DEF_MAX,POS_DPD_COUNT,POS_DPD_RATE
0,100001,9,-96,4.000000,1.444444,0.777778,7,0.777778,7,1,0.111111
1,100002,19,-19,24.000000,15.000000,0.000000,0,0.000000,0,0,0.000000
2,100003,28,-77,10.107143,5.785714,0.000000,0,0.000000,0,0,0.000000
3,100004,4,-27,3.750000,2.250000,0.000000,0,0.000000,0,0,0.000000
4,100005,11,-25,11.700000,7.200000,0.000000,0,0.000000,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
337247,456251,9,-9,7.875000,4.375000,0.000000,0,0.000000,0,0,0.000000
337248,456252,7,-82,6.000000,3.000000,0.000000,0,0.000000,0,0,0.000000
337249,456253,17,-96,6.705882,2.000000,0.294118,5,0.294118,5,1,0.058824
337250,456254,20,-11,14.900000,10.350000,0.000000,0,0.000000,0,0,0.000000


9.8 Credit Card Balance Aggregation

A data frame that provides information on how a customer uses their bank cards.

In [172]:
credit_card_df["CC_UTILIZATION"] = safe_divide(credit_card_df["AMT_BALANCE"],credit_card_df["AMT_CREDIT_LIMIT_ACTUAL"])
credit_card_df["CC_DPD_FLAG"] = (credit_card_df["SK_DPD"] > 0).astype(int)

In [173]:
credit_card_customer_agg = (
    credit_card_df
    .groupby("SK_ID_CURR")
    .agg(
        CC_RECORD_COUNT=("SK_ID_PREV","count"),
        CC_BALANCE_MEAN=("AMT_BALANCE","mean"),
        CC_BALANCE_MAX=("AMT_BALANCE","max"),
        CC_CREDIT_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL","mean"),
        CC_DRAWINGS_TOTAL=("AMT_DRAWINGS_CURRENT","sum"),
        CC_PAYMENT_TOTAL=("AMT_PAYMENT_TOTAL_CURRENT","sum"),
        CC_UTILIZATION_MEAN=("CC_UTILIZATION","mean"),
        CC_UTILIZATION_MAX=("CC_UTILIZATION","max"),
        CC_DPD_MEAN=("SK_DPD","mean"),
        CC_DPD_MAX=("SK_DPD","max"),
        CC_DPD_COUNT=("CC_DPD_FLAG","sum")
    )
    .reset_index()
)

In [174]:
credit_card_customer_agg["CC_DPD_RATE"] = safe_divide(credit_card_customer_agg["CC_DPD_COUNT"], credit_card_customer_agg["CC_RECORD_COUNT"])

In [175]:
credit_card_customer_agg

,SK_ID_CURR,CC_RECORD_COUNT,CC_BALANCE_MEAN,CC_BALANCE_MAX,CC_CREDIT_LIMIT_MEAN,CC_DRAWINGS_TOTAL,CC_PAYMENT_TOTAL,CC_UTILIZATION_MEAN,CC_UTILIZATION_MAX,CC_DPD_MEAN,CC_DPD_MAX,CC_DPD_COUNT,CC_DPD_RATE
0,100006,6,0.000000,0.000,270000.000000,0.000,0.000,0.000000,0.000000,0.000000,0,0,0.000000
1,100011,74,54482.111149,189000.000,164189.189189,180000.000,334485.000,0.302678,1.050000,0.000000,0,0,0.000000
2,100013,96,18159.919219,161420.220,131718.750000,571500.000,654448.545,0.115301,1.024890,0.010417,1,1,0.010417
3,100021,17,0.000000,0.000,675000.000000,0.000,0.000,0.000000,0.000000,0.000000,0,0,0.000000
4,100023,8,0.000000,0.000,135000.000000,0.000,0.000,0.000000,0.000000,0.000000,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103553,456244,41,131834.730732,453627.675,296341.463415,1100537.910,1341542.340,0.444874,1.008061,0.000000,0,0,0.000000
103554,456246,8,13136.731875,43490.115,135000.000000,121594.050,124434.720,0.097309,0.322149,0.000000,0,0,0.000000
103555,456247,95,23216.396211,190202.130,144000.000000,204203.115,391008.420,0.161225,1.056679,0.031579,1,3,0.031579
103556,456248,23,0.000000,0.000,900000.000000,0.000,0.000,0.000000,0.000000,0.000000,0,0,0.000000


9.9 Merge Aggregated Features

In [176]:
#Verify that all df files have only one SK_ID_CURR per row
aggregated_tables = {
    "bureau": bureau_customer_agg,
    "previous_application": previous_customer_agg,
    "installments": installments_customer_agg,
    "pos_cash": pos_customer_agg,
    "credit_card": credit_card_customer_agg
}

In [177]:
for name, df in aggregated_tables.items():
    duplicated_ids = df["SK_ID_CURR"].duplicated().sum()
    print(name, "duplicated SK_ID_CURR:", duplicated_ids)

bureau duplicated SK_ID_CURR: 0
previous_application duplicated SK_ID_CURR: 0
installments duplicated SK_ID_CURR: 0
pos_cash duplicated SK_ID_CURR: 0
credit_card duplicated SK_ID_CURR: 0


In [178]:
#Add history indicators; otherwise, NaN values appear when performing a left join
bureau_customer_agg["HAS_BUREAU_HISTORY"] = 1
previous_customer_agg["HAS_PREVIOUS_APPLICATION"] = 1
installments_customer_agg["HAS_INSTALLMENT_HISTORY"] = 1
pos_customer_agg["HAS_POS_HISTORY"] = 1
credit_card_customer_agg["HAS_CREDIT_CARD_HISTORY"] = 1

In [179]:
#Merge for each df
for name, aggregated_df in aggregated_tables.items():
    advanced_train_df = (
        advanced_train_df.merge(
            aggregated_df,
            on="SK_ID_CURR",
            how="left",
            validate="one_to_one"
        )
    )
    print(name, advanced_train_df.shape)

bureau (307511, 155)
previous_application (307511, 169)
installments (307511, 179)
pos_cash (307511, 190)
credit_card (307511, 203)


9.10 Missing Historical Information

After merging each df, new NaN values have appeared that need to be handled.

In [180]:
#Fill History Flags
history_flags = [
    "HAS_BUREAU_HISTORY",
    "HAS_PREVIOUS_APPLICATION",
    "HAS_INSTALLMENT_HISTORY",
    "HAS_POS_HISTORY",
    "HAS_CREDIT_CARD_HISTORY"
]

advanced_train_df[history_flags] = (advanced_train_df[history_flags].fillna(0).astype(int))

In [181]:
#Fill structural counts with 0
historical_count_features = [
    "BUREAU_CREDIT_COUNT",
    "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_COUNT",
    "PREV_APPLICATION_COUNT",
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "INSTALMENT_RECORD_COUNT",
    "LATE_PAYMENT_COUNT",
    "POS_RECORD_COUNT",
    "POS_DPD_COUNT",
    "CC_RECORD_COUNT",
    "CC_DPD_COUNT"
]

advanced_train_df[historical_count_features] = (advanced_train_df[historical_count_features].fillna(0))

9.11 Update Feature Metadata

In [182]:
auxiliary_features = [feature for feature in advanced_train_df.columns if feature not in prepared_train_df.columns]
print("New auxiliary features:",len(auxiliary_features))

New auxiliary features: 66


In [183]:
def classify_auxiliary_feature(feature):
    if feature.startswith("HAS_"):
        return "Binary / Flag"
    if "COUNT" in feature and "RATE" not in feature:
        return "Numerical Count"
    return "Numerical Continuous"

In [184]:
auxiliary_metadata = pd.DataFrame({"Feature": auxiliary_features})

auxiliary_metadata["Pandas Type"] = [str(advanced_train_df[feature].dtype)for feature in auxiliary_features]
auxiliary_metadata["Analytical Type"] = auxiliary_metadata["Feature"].apply(classify_auxiliary_feature)

In [185]:
auxiliary_metadata

,Feature,Pandas Type,Analytical Type
0,BUREAU_CREDIT_COUNT,float64,Numerical Count
1,BUREAU_ACTIVE_COUNT,float64,Numerical Count
2,BUREAU_CLOSED_COUNT,float64,Numerical Count
3,BUREAU_CREDIT_SUM_TOTAL,float64,Numerical Continuous
4,BUREAU_CREDIT_SUM_MEAN,float64,Numerical Continuous
...,...,...,...
61,CC_DPD_MEAN,float64,Numerical Continuous
62,CC_DPD_MAX,float64,Numerical Continuous
63,CC_DPD_COUNT,float64,Numerical Count
64,CC_DPD_RATE,float64,Numerical Continuous


In [186]:
data_types_advanced = pd.concat([data_types, auxiliary_metadata],ignore_index=True)

data_types_advanced.to_csv("../outputs/data_preparation/data_types_advanced.csv", index=False)

9.12 Integration Log

In [187]:
auxiliary_integration_log = pd.DataFrame({
    "Source Table": [
        "bureau + bureau_balance",
        "previous_application",
        "installments_payments",
        "POS_CASH_balance",
        "credit_card_balance"
    ],

    "Aggregation Level": [
        "SK_ID_CURR",
        "SK_ID_CURR",
        "SK_ID_CURR",
        "SK_ID_CURR",
        "SK_ID_CURR"
    ],

    "Purpose": [
        "Previous external credit history and delinquency",
        "Previous Home Credit application history",
        "Historical payment behaviour",
        "POS loan repayment and delinquency history",
        "Credit card utilisation and delinquency history"
    ]
})

In [188]:
source_prefixes = {
    "bureau + bureau_balance": ["BUREAU_"],
    "previous_application": ["PREV_"],
    "installments_payments": ["INSTALMENT_", "PAYMENT_", "LATE_PAYMENT_"],
    "POS_CASH_balance": ["POS_"],
    "credit_card_balance": ["CC_"]
}

In [189]:
auxiliary_integration_log.to_csv("../outputs/data_preparation/auxiliary_integration_log.csv", index=False)

Now I have:

prepared_train_df → data_types_prepared.csv → baseline (without tablas auxiliares)

advanced_train_df → data_types_advanced.csv → advanced (with tablas auxiliares)

In [190]:
print("Baseline rows:", len(prepared_train_df))
print("Advanced rows:", len(advanced_train_df))

Baseline rows: 307511
Advanced rows: 307511


In [191]:
print("=== BASELINE ===")
print(
    "Shape:",
    prepared_train_df.shape
)

print(
    "Unique applicants:",
    prepared_train_df[
        "SK_ID_CURR"
    ].nunique()
)

print()


print("=== ADVANCED ===")
print(
    "Shape:",
    advanced_train_df.shape
)

print(
    "Unique applicants:",
    advanced_train_df[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "Auxiliary features added:",
    len(auxiliary_features)
)

print()


print("=== METADATA ===")
print(
    "Baseline metadata:",
    len(data_types)
)

print(
    "Advanced metadata:",
    len(data_types_advanced)
)

=== BASELINE ===
Shape: (307511, 137)
Unique applicants: 307511

=== ADVANCED ===
Shape: (307511, 203)
Unique applicants: 307511
Auxiliary features added: 66

=== METADATA ===
Baseline metadata: 137
Advanced metadata: 203


## 10. Train / Validation / Test Split

10.1 Split Strategy

The ‘application_test.csv’ file will be used as an external test set for the models -> Final external prediction.

The ‘application_train.csv’ file is the one we will split into train, validation, and internal test sets for the models.

The purpose of each of these subfiles is:

- Train: Adjust preprocessing and train the models (70% of total data)

- Validation: Model selection and hyperparameter tuning (15% of total data)

- Test: Final unbiased evaluation of the models (15% of total data)

In [192]:
#Establish the foundations for reproducibility
RANDOM_STATE = 42

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

assert (TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == 1.0)

In [193]:
assert (TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == 1.0)

10.2 Separate Identifier, Target and Predictors

In [194]:
#Separar las columnas SK_ID_CURR y TARGET
ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"

split_df = prepared_train_df[[ID_COLUMN, TARGET_COLUMN]].copy()

split_df.head()

,SK_ID_CURR,TARGET
0,100002,1
1,100003,0
2,100004,0
3,100006,0
4,100007,0


10.3 Create Stratified Internal Split

The dataset is imbalanced with respect to the TARGET variable:

- TARGET = 0 → 91.9%

- TARGET = 1 → 8.1%

In [195]:
from sklearn.model_selection import train_test_split

In [196]:
#Split the dataframe into 70% training and 30% testing in a balanced way using the TARGET variable
train_split_df, temp_split_df = train_test_split(
    split_df,
    test_size=(VALIDATION_SIZE + TEST_SIZE),stratify=split_df[TARGET_COLUMN],random_state=RANDOM_STATE
)

In [197]:
print("Train:", train_split_df.shape)
print("Temporary:", temp_split_df.shape)

Train: (215257, 2)
Temporary: (92254, 2)


In [198]:
#Split 30% of the test dataframe into 15% for testing and 15% for validation
validation_split_df, test_split_df = train_test_split(
    temp_split_df,
    test_size=0.50,
    stratify=temp_split_df[TARGET_COLUMN],
    random_state=RANDOM_STATE
)

In [199]:
#Check Sizes and Percentages
print("Train rows:", len(train_split_df))
print("Validation rows:", len(validation_split_df))
print("Test rows:", len(test_split_df))

total_rows = len(split_df)

print("Train %:", round(len(train_split_df) / total_rows * 100, 2))
print("Validation %:", round(len(validation_split_df) / total_rows * 100, 2))
print("Test %:", round(len(test_split_df) / total_rows * 100, 2))

Train rows: 215257
Validation rows: 46127
Test rows: 46127
Train %: 70.0
Validation %: 15.0
Test %: 15.0


10.4 Verify Target Distribution

The purpose of using stratify is to preserve the imbalance in the TARGET variable.

In [200]:
def target_distribution(df):
    distribution = df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100
    return distribution

In [201]:
print("Full dataset")
print(target_distribution(split_df))

print("\nTrain")
print(target_distribution(train_split_df))

print("\nValidation")
print(target_distribution(validation_split_df))

print("\nTest")
print(target_distribution(test_split_df))

Full dataset
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

Train
TARGET
0    91.927324
1     8.072676
Name: proportion, dtype: float64

Validation
TARGET
0    91.926637
1     8.073363
Name: proportion, dtype: float64

Test
TARGET
0    91.926637
1     8.073363
Name: proportion, dtype: float64


In [202]:
split_target_distribution = pd.DataFrame({
    "Full": target_distribution(split_df),
    "Train": target_distribution(train_split_df),
    "Validation": target_distribution(validation_split_df),
    "Test": target_distribution(test_split_df)
}).T

split_target_distribution.columns = ["TARGET_0_Percentage", "TARGET_1_Percentage"]
split_target_distribution

,TARGET_0_Percentage,TARGET_1_Percentage
Full,91.927118,8.072882
Train,91.927324,8.072676
Validation,91.926637,8.073363
Test,91.926637,8.073363


10.5 Apply the Same Split to Baseline and Advanced Datasets

In [203]:
train_ids = set(train_split_df[ID_COLUMN])
validation_ids = set(validation_split_df[ID_COLUMN])
test_ids = set(test_split_df[ID_COLUMN])

In [204]:
#Baseline dataframe splits
baseline_train_df = prepared_train_df[prepared_train_df[ID_COLUMN].isin(train_ids)].copy()
baseline_validation_df = prepared_train_df[prepared_train_df[ID_COLUMN].isin(validation_ids)].copy()
baseline_test_df = prepared_train_df[prepared_train_df[ID_COLUMN].isin(test_ids)].copy()

In [205]:
#Advance dataframe splits
advanced_train_split_df = advanced_train_df[advanced_train_df[ID_COLUMN].isin(train_ids)].copy()
advanced_validation_df = advanced_train_df[advanced_train_df[ID_COLUMN].isin(validation_ids)].copy()
advanced_test_df = advanced_train_df[advanced_train_df[ID_COLUMN].isin(test_ids)].copy()

In [206]:
# Balance the train set only by undersampling with TARGET = 0. train_df and prepared_train_df remain unchanged.
class_counts = train_split_df[TARGET_COLUMN].value_counts()

if not {0, 1}.issubset(class_counts.index):
    raise ValueError("El conjunto de entrenamiento debe contener las clases 0 y 1.")

samples_per_class = int(class_counts.min())

balanced_train_split_df = (
    train_split_df
    .groupby(TARGET_COLUMN, group_keys=False)
    .sample(n=samples_per_class, random_state=RANDOM_STATE)
)

train_split_df = balanced_train_split_df.copy()
train_ids = set(train_split_df[ID_COLUMN])

baseline_train_df = baseline_train_df[
    baseline_train_df[ID_COLUMN].isin(train_ids)
].copy()

advanced_train_split_df = advanced_train_split_df[
    advanced_train_split_df[ID_COLUMN].isin(train_ids)
].copy()

split_target_distribution = pd.DataFrame({
    "Full": target_distribution(split_df),
    "Train": target_distribution(train_split_df),
    "Validation": target_distribution(validation_split_df),
    "Test": target_distribution(test_split_df)
}).T
split_target_distribution.columns = ["TARGET_0_Percentage", "TARGET_1_Percentage"]

print("Distribución de TARGET en Train después del balanceo:")
print(train_split_df[TARGET_COLUMN].value_counts().sort_index())

assert train_split_df[TARGET_COLUMN].value_counts().nunique() == 1
print("\nTrain balanceado correctamente.")
print("Validation y Test mantienen su distribución original.")

Distribución de TARGET en Train después del balanceo:
TARGET
0    17377
1    17377
Name: count, dtype: int64

Train balanceado correctamente.
Validation y Test mantienen su distribución original.


10.6 Create X and Y objects to train

In [207]:
#Baseline Y
y_train = baseline_train_df[TARGET_COLUMN].copy()
y_validation = baseline_validation_df[TARGET_COLUMN].copy()
y_test = baseline_test_df[TARGET_COLUMN].copy()

In [208]:
#Baseline X
X_train_baseline = baseline_train_df.drop(columns=[ID_COLUMN, TARGET_COLUMN])
X_validation_baseline = (baseline_validation_df.drop(columns=[ID_COLUMN, TARGET_COLUMN]))
X_test_baseline = baseline_test_df.drop(columns=[ID_COLUMN, TARGET_COLUMN])

In [209]:
#Advanced X
X_train_advanced = (advanced_train_split_df.drop(columns=[ID_COLUMN, TARGET_COLUMN]))
X_validation_advanced = (advanced_validation_df.drop(columns=[ID_COLUMN, TARGET_COLUMN]))
X_test_advanced = (advanced_test_df.drop(columns=[ID_COLUMN, TARGET_COLUMN]))

In [210]:
#Check shapes
print("BASELINE")
print("X_train:", X_train_baseline.shape)
print("X_validation:", X_validation_baseline.shape)
print("X_test:", X_test_baseline.shape)

print()

print("ADVANCED")
print("X_train:", X_train_advanced.shape)
print("X_validation:", X_validation_advanced.shape)
print("X_test:", X_test_advanced.shape)

BASELINE
X_train: (34754, 135)
X_validation: (46127, 135)
X_test: (46127, 135)

ADVANCED
X_train: (34754, 201)
X_validation: (46127, 201)
X_test: (46127, 201)


10.7 Save Split IDs

In [211]:
#Save the IDs for reproducibility
train_ids_df = train_split_df[[ID_COLUMN, TARGET_COLUMN]].sort_values(ID_COLUMN)
validation_ids_df = validation_split_df[[ID_COLUMN, TARGET_COLUMN]].sort_values(ID_COLUMN)
test_ids_df = test_split_df[[ID_COLUMN, TARGET_COLUMN]].sort_values(ID_COLUMN)

In [212]:
train_ids_df.to_csv("../data/processed/train_ids.csv", index=False)
validation_ids_df.to_csv("../data/processed/validation_ids.csv", index=False)
test_ids_df.to_csv("../data/processed/test_ids.csv", index=False)

10.8 Split Summary

In [213]:
split_summary = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Test"
    ],

    "Rows": [
        len(train_split_df),
        len(validation_split_df),
        len(test_split_df)
    ],

    "Percentage": [
        len(train_split_df) / len(split_df) * 100,
        len(validation_split_df) / len(split_df) * 100,
        len(test_split_df) / len(split_df) * 100
    ],

    "TARGET_1_Count": [
        train_split_df[TARGET_COLUMN].sum(),
        validation_split_df[TARGET_COLUMN].sum(),
        test_split_df[TARGET_COLUMN].sum()
    ],

    "TARGET_1_Rate": [
        train_split_df[TARGET_COLUMN].mean() * 100,
        validation_split_df[TARGET_COLUMN].mean() * 100,
        test_split_df[TARGET_COLUMN].mean() * 100
    ]
})
split_summary

,Split,Rows,Percentage,TARGET_1_Count,TARGET_1_Rate
0,Train,34754,11.301710,17377,50.000000
1,Validation,46127,15.000114,3724,8.073363
2,Test,46127,15.000114,3724,8.073363


In [214]:
split_summary.to_csv("../outputs/data_preparation/split_summary.csv", index=False)
split_target_distribution.to_csv("../outputs/data_preparation/split_target_distribution.csv")

## 11. Leakage Prevention and Preprocessing Pipeline

11.1 Preprocessing and Leakage Strategy

From now on, our rule is:

- Training dataframe -> FIT

- Validation dataframe -> TRANSFORM ONLY

- Test dataframe -> TRANSFORM ONLY

11.2 Train-Only Feature Selection

Previously saw that I had constant variables; now that I have the dataframes split up, I can work on them. I’ll do this separately for the baseline and advanced models.

Furthermore, since these variables are continuous and have the same value in every record, they offer no advantage in training the models.

In [215]:
def find_constant_features(X):
    return [feature for feature in X.columns if X[feature].nunique(dropna=False) <= 1]

In [216]:
baseline_constant_features = find_constant_features(X_train_baseline)
advanced_constant_features = find_constant_features(X_train_advanced)

In [217]:
def find_near_constant_features(X, threshold=99):
    results = []
    for feature in X.columns:
        value_counts = X[feature].value_counts(normalize=True, dropna=False)

        if len(value_counts) == 0:
            continue

        dominant_percentage = value_counts.iloc[0] * 100

        if dominant_percentage >= threshold:
            results.append({"Feature": feature, "Dominant Value Percentage": dominant_percentage})
    return pd.DataFrame(results)

In [218]:
baseline_near_constant = find_near_constant_features(X_train_baseline)
advanced_near_constant = find_near_constant_features(X_train_advanced)

In [219]:
baseline_near_constant

,Feature,Dominant Value Percentage
0,FLAG_MOBIL,100.000000
1,FLAG_CONT_MOBILE,99.792830
2,FLAG_DOCUMENT_2,99.991368
3,FLAG_DOCUMENT_4,99.997123
4,FLAG_DOCUMENT_7,99.976981
5,FLAG_DOCUMENT_9,99.674858
6,FLAG_DOCUMENT_10,100.000000
7,FLAG_DOCUMENT_11,99.651839
8,FLAG_DOCUMENT_12,100.000000
9,FLAG_DOCUMENT_13,99.766933


In [220]:
advanced_near_constant

,Feature,Dominant Value Percentage
0,FLAG_MOBIL,100.000000
1,FLAG_CONT_MOBILE,99.792830
2,FLAG_DOCUMENT_2,99.991368
3,FLAG_DOCUMENT_4,99.997123
4,FLAG_DOCUMENT_7,99.976981
5,FLAG_DOCUMENT_9,99.674858
6,FLAG_DOCUMENT_10,100.000000
7,FLAG_DOCUMENT_11,99.651839
8,FLAG_DOCUMENT_12,100.000000
9,FLAG_DOCUMENT_13,99.766933


In [221]:
def find_high_correlation_features(X, metadata, threshold=0.98):
    numerical_types = [
        "Numerical Continuous",
        "Numerical Count",
        "Temporal / Days",
        "External Score",
        "Ordinal"
    ]

    numerical_features = metadata.loc[metadata["Analytical Type"].isin(numerical_types), "Feature"].tolist()
    numerical_features = [feature for feature in numerical_features if feature in X.columns and pd.api.types.is_numeric_dtype(X[feature])]

    correlation_matrix = X[numerical_features].corr(method="pearson").abs()

    upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

    correlation_pairs = upper_triangle.stack().reset_index()
    correlation_pairs.columns = ["Feature 1", "Feature 2", "Absolute Correlation"]
    correlation_pairs = correlation_pairs[correlation_pairs["Absolute Correlation"] >= threshold].sort_values("Absolute Correlation", ascending=False).reset_index(drop=True)
    
    return correlation_pairs

In [222]:
baseline_high_corr_pairs = find_high_correlation_features(X_train_baseline, data_types, threshold=0.98)
advanced_high_corr_pairs = find_high_correlation_features(X_train_advanced, data_types_advanced, threshold=0.98)

I eliminate the variables that do not meet the following criteria:

1. Keep the one with the fewest missing values.
2. If they have the same number of missing values: prefer AVG over MEDI/MODE.
3. If they are still tied: keep Feature 1 and eliminate Feature 2.

In [223]:
def feature_preference_score(feature):
    if feature.endswith("_AVG"):
        return 3
    elif feature.endswith("_MEDI"):
        return 2
    elif feature.endswith("_MODE"):
        return 1
    else:
        return 2

In [224]:
def select_correlated_features_to_drop(X_train, correlation_pairs):
    missing_rate = X_train.isna().mean()

    dropped = set()
    decisions = []
    for _, row in correlation_pairs.iterrows():
        feature_1 = row["Feature 1"]
        feature_2 = row["Feature 2"]
        correlation = row["Absolute Correlation"]

        # One already removed
        if feature_1 in dropped or feature_2 in dropped:
            continue
        missing_1 = missing_rate[feature_1]
        missing_2 = missing_rate[feature_2]

        # Missingness criterion
        if missing_1 > missing_2:
            remove = feature_1
            keep = feature_2
            reason = "Higher missingness"
        elif missing_2 > missing_1:
            remove = feature_2
            keep = feature_1
            reason = "Higher missingness"
        else:
            preference_1 = feature_preference_score(feature_1)
            preference_2 = feature_preference_score(feature_2)

            if preference_1 > preference_2:
                keep = feature_1
                remove = feature_2
                reason = "Preferred representation"
            elif preference_2 > preference_1:
                keep = feature_2
                remove = feature_1
                reason = "Preferred representation"
            else:
                keep = feature_1
                remove = feature_2
                reason = "Equivalent redundancy; second feature removed"
        dropped.add(remove)

        decisions.append({
            "Feature Kept": keep,
            "Feature Removed": remove,
            "Absolute Correlation": correlation,
            "Missing Kept": missing_rate[keep] * 100,
            "Missing Removed": missing_rate[remove] * 100,
            "Reason": reason
        })

    return (sorted(dropped), pd.DataFrame(decisions))

In [225]:
#Baseline
baseline_corr_features_to_drop, baseline_correlation_decisions = select_correlated_features_to_drop(X_train_baseline, baseline_high_corr_pairs)

#Advanced
advanced_corr_features_to_drop, advanced_correlation_decisions = select_correlated_features_to_drop(X_train_advanced, advanced_high_corr_pairs)

In [226]:
#Comprobar
print("Baseline correlated features removed:", len(baseline_corr_features_to_drop))
print("Advanced correlated features removed:", len(advanced_corr_features_to_drop))

Baseline correlated features removed: 21
Advanced correlated features removed: 23


In [227]:
#Create Final Deletion Lists
baseline_features_to_drop = sorted(set(baseline_constant_features) | set(baseline_corr_features_to_drop))
advanced_features_to_drop = sorted(set(advanced_constant_features) | set(advanced_corr_features_to_drop))

In [228]:
#Aply to X baseline
X_train_baseline_selected = X_train_baseline.drop(columns=baseline_features_to_drop)
X_validation_baseline_selected = X_validation_baseline.drop(columns=baseline_features_to_drop)
X_test_baseline_selected = X_test_baseline.drop(columns=baseline_features_to_drop)

In [229]:
#Aply to X advanced
X_train_advanced_selected = X_train_advanced.drop(columns=advanced_features_to_drop)
X_validation_advanced_selected = X_validation_advanced.drop(columns=advanced_features_to_drop)
X_test_advanced_selected = X_test_advanced.drop(columns=advanced_features_to_drop)

In [230]:
#Save
baseline_correlation_decisions.to_csv("../outputs/data_preparation/baseline_correlation_selection.csv", index=False)
advanced_correlation_decisions.to_csv("../outputs/data_preparation/advanced_correlation_selection.csv", index=False)
baseline_near_constant.to_csv("../outputs/data_preparation/baseline_near_constant_review.csv", index=False)
advanced_near_constant.to_csv("../outputs/data_preparation/advanced_near_constant_review.csv", index=False)

11.3 Final Predictor Groups

In [231]:
def get_feature_groups(X_train, metadata):
    available_features = set(X_train.columns)

    def get_group(analytical_type):
        return [feature for feature in metadata.loc[metadata["Analytical Type"] == analytical_type, "Feature"].tolist() if feature in available_features]

    return {
        "continuous": get_group("Numerical Continuous"),
        "count": get_group("Numerical Count"),
        "external":get_group("External Score"),
        "categorical": get_group("Categorical"),
        "binary": get_group("Binary / Flag"),
        "ordinal": get_group("Ordinal")
    }

In [232]:
baseline_groups = get_feature_groups(X_train_baseline_selected, data_types)
advanced_groups = get_feature_groups(X_train_advanced_selected, data_types_advanced)

In [233]:
def split_numeric_text(X, features):
    numeric = []
    text = []
    for feature in features:
        if pd.api.types.is_numeric_dtype(X[feature]):
            numeric.append(feature)
        else:
            text.append(feature)

    return numeric, text

In [234]:
#Baseline
baseline_binary_numeric, baseline_binary_text = split_numeric_text(X_train_baseline_selected, baseline_groups["binary"])

#Advanced
advanced_binary_numeric, advanced_binary_text = split_numeric_text(X_train_advanced_selected, baseline_groups["binary"])

In [235]:
#Baseline
baseline_ordinal_numeric, baseline_ordinal_text = split_numeric_text(X_train_baseline_selected, baseline_groups["ordinal"])

#Advanced
advanced_ordinal_numeric, advanced_ordinal_text = split_numeric_text(X_train_advanced_selected, advanced_groups["ordinal"])

11.4 Numerical Missing-Value Preprocessing

In [236]:
#Numerical pipeline
numerical_scaled_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

In [237]:
#Numerical pipeline without scaling
numerical_unscaled_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

In [238]:
#External scores
external_score_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median", add_indicator=True))]
)

11.5 Categorical Preprocessing

For nominal categorical variables:

Missing → explicit “Missing”

↓

One-Hot Encoding

In [239]:
categorical_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

11.6 Binary and Ordinal Preprocessing

In [240]:
#Numeric binary features -> 0/1
binary_numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent"))]
)

In [241]:
#Text binary features -> Y/N. OrdenalEncoder learn with this: Y->1 y N->0.
binary_text_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]
)

In [242]:
#Numerical ordinal -> 1, 2, 3...
ordinal_numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent"))]
)

In [243]:
#Text ordinal features
ordinal_category_orders = {
    "NAME_EDUCATION_TYPE": [
        "Unknown",
        "Lower secondary",
        "Secondary / secondary special",
        "Incomplete higher",
        "Higher education",
        "Academic degree"
    ]
}

undefined_ordinal_features = [feature for feature in baseline_ordinal_text if feature not in ordinal_category_orders]
undefined_ordinal_features

[]

Returning an empty list is a good sign, since it means we don't have any textual ordinal variables with an undefined order.

In [244]:
#Create categories with the same column order
#Baseline
baseline_ordinal_categories = [ordinal_category_orders[feature] for feature in baseline_ordinal_text]
baseline_ordinal_text_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OrdinalEncoder(categories=baseline_ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1))
    ]
)

#Advanced
advanced_ordinal_categories = [ordinal_category_orders[feature] for feature in advanced_ordinal_text]
advanced_ordinal_text_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OrdinalEncoder(categories=advanced_ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1))
    ]
)

11.7 Model-Specific Scaling Pipelines

In Section 6, we established that not all models will undergo the same preprocessing, so we divided them into two groups:

SCALED

→ Logistic Regression

→ KNN

→ SVM

→ Neural Network

UNSCALED

→ Decision Tree

→ Random Forest

→ Gradient Boosting

→ XGBoost / LightGBM


In [245]:
#ColumnTransformer
def build_preprocessor(X_train, groups, scale_numerical, ordinal_text_pipeline):
    binary_numeric, binary_text = split_numeric_text(X_train, groups["binary"])
    ordinal_numeric, ordinal_text = split_numeric_text(X_train, groups["ordinal"])
    numerical_features = groups["continuous"] + groups["count"]

    if scale_numerical:
        numerical_pipeline = numerical_scaled_pipeline
    else:
        numerical_pipeline = numerical_unscaled_pipeline

    transformers = []
    # Numerical continuous/count
    if numerical_features:
        transformers.append(("numerical", numerical_pipeline, numerical_features))

    # External scores
    if groups["external"]:
        transformers.append(("external_scores", external_score_pipeline, groups["external"]))

    # Nominal categorical
    if groups["categorical"]:
        transformers.append(("categorical", categorical_pipeline, groups["categorical"]))

    # Binary numeric
    if binary_numeric:
        transformers.append(("binary_numeric", binary_numeric_pipeline, binary_numeric))

    # Binary text
    if binary_text:
        transformers.append(("binary_text", binary_text_pipeline, binary_text))

    # Ordinal numeric
    if ordinal_numeric:
        transformers.append(("ordinal_numeric", ordinal_numeric_pipeline, ordinal_numeric))

    # Ordinal text
    if ordinal_text:
        transformers.append(("ordinal_text", ordinal_text_pipeline, ordinal_text))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

11.8 Fit Baseline Preprocessors

In [246]:
#Scaled Baseline
baseline_preprocessor_scaled = build_preprocessor(
    X_train=X_train_baseline_selected,
    groups=baseline_groups,
    scale_numerical=True,
    ordinal_text_pipeline=baseline_ordinal_text_pipeline
)

In [247]:
#Unscaled Baseline
baseline_preprocessor_unscaled = build_preprocessor(
    X_train=X_train_baseline_selected,
    groups=baseline_groups,
    scale_numerical=False,
    ordinal_text_pipeline=baseline_ordinal_text_pipeline
)

In [248]:
#Data scaling and .fit -> Results: median learning, category learning, one-hot learning, mean/std learning
X_train_baseline_scaled = baseline_preprocessor_scaled.fit_transform(X_train_baseline_selected)
X_train_baseline_unscaled = baseline_preprocessor_unscaled.fit_transform(X_train_baseline_selected)

11.9 Fit Advanced Preprocessors

In [249]:
#Scaled Advanced
advanced_preprocessor_scaled = build_preprocessor(
    X_train=X_train_advanced_selected,
    groups=advanced_groups,
    scale_numerical=True,
    ordinal_text_pipeline=advanced_ordinal_text_pipeline
)

In [250]:
#Unscaled Advanced
advanced_preprocessor_unscaled = build_preprocessor(
    X_train=X_train_advanced_selected,
    groups=advanced_groups,
    scale_numerical=False,
    ordinal_text_pipeline=advanced_ordinal_text_pipeline
)

In [251]:
#Data Scaling and .fit
X_train_advanced_scaled = advanced_preprocessor_scaled.fit_transform(X_train_advanced_selected)
X_train_advanced_unscaled = advanced_preprocessor_unscaled.fit_transform(X_train_advanced_selected)

11.10 Transform Validation and Test Sets

In [252]:
#Baseline Scaled
X_validation_baseline_scaled = baseline_preprocessor_scaled.transform(X_validation_baseline_selected)
X_test_baseline_scaled = baseline_preprocessor_scaled.transform(X_test_baseline_selected)

In [253]:
#Baseline Unscaled
X_validation_baseline_unscaled = baseline_preprocessor_unscaled.transform(X_validation_baseline_selected)
X_test_baseline_unscaled = baseline_preprocessor_unscaled.transform(X_test_baseline_selected)

In [254]:
#Advanced Scaled
X_validation_advanced_scaled = advanced_preprocessor_scaled.transform(X_validation_advanced_selected)
X_test_advanced_scaled = advanced_preprocessor_scaled.transform(X_test_advanced_selected)

In [255]:
#Advanced Unscaled
X_validation_advanced_unscaled = advanced_preprocessor_unscaled.transform(X_validation_advanced_selected)
X_test_advanced_unscaled = advanced_preprocessor_unscaled.transform(X_test_advanced_selected)

11.11 Feature Names After Encoding

After one-hot encoding, the number of features will have increased.
I need to know exactly which ones exist.

In [256]:
#Baseline Scaled
baseline_scaled_feature_names = baseline_preprocessor_scaled.get_feature_names_out()
#Baseline Unscaled
baseline_unscaled_feature_names = baseline_preprocessor_unscaled.get_feature_names_out()
#Advanced Scaled
advanced_scaled_feature_names = advanced_preprocessor_scaled.get_feature_names_out()
#Advanced Unscaled
advanced_unscaled_feature_names = advanced_preprocessor_unscaled.get_feature_names_out()

In [257]:
print("Baseline predictors before preprocessing:", X_train_baseline_selected.shape[1])
print("Baseline predictors after preprocessing:", len(baseline_scaled_feature_names))
print("Advanced predictors before preprocessing:", X_train_advanced_selected.shape[1])
print("Advanced predictors after preprocessing:", len(advanced_scaled_feature_names))

Baseline predictors before preprocessing: 111
Baseline predictors after preprocessing: 234
Advanced predictors before preprocessing: 175
Advanced predictors after preprocessing: 298


In [258]:
#Save feature names
pd.DataFrame({"Feature": baseline_scaled_feature_names}).to_csv("../outputs/data_preparation/baseline_preprocessed_features.csv", index=False)
pd.DataFrame({"Feature": advanced_scaled_feature_names}).to_csv("../outputs/data_preparation/advanced_preprocessed_features.csv", index=False)

In [259]:
preprocessing_summary = pd.DataFrame({
    "Dataset": [
        "Baseline Scaled",
        "Baseline Unscaled",
        "Advanced Scaled",
        "Advanced Unscaled"
    ],

    "Train Rows": [
        X_train_baseline_scaled.shape[0],
        X_train_baseline_unscaled.shape[0],
        X_train_advanced_scaled.shape[0],
        X_train_advanced_unscaled.shape[0]
    ],

    "Validation Rows": [
        X_validation_baseline_scaled.shape[0],
        X_validation_baseline_unscaled.shape[0],
        X_validation_advanced_scaled.shape[0],
        X_validation_advanced_unscaled.shape[0]
    ],

    "Test Rows": [
        X_test_baseline_scaled.shape[0],
        X_test_baseline_unscaled.shape[0],
        X_test_advanced_scaled.shape[0],
        X_test_advanced_unscaled.shape[0]
    ],

    "Final Features": [
        X_train_baseline_scaled.shape[1],
        X_train_baseline_unscaled.shape[1],
        X_train_advanced_scaled.shape[1],
        X_train_advanced_unscaled.shape[1]
    ]
})

In [260]:
preprocessing_summary

,Dataset,Train Rows,Validation Rows,Test Rows,Final Features
0,Baseline Scaled,34754,46127,46127,234
1,Baseline Unscaled,34754,46127,46127,234
2,Advanced Scaled,34754,46127,46127,298
3,Advanced Unscaled,34754,46127,46127,298


In [261]:
preprocessing_summary.to_csv("../outputs/data_preparation/preprocessing_summary.csv", index=False)

## 12. Final Dataset Summary

12.1 Final Dataset Inventory

In [262]:
final_dataset_inventory = pd.DataFrame({
    "Dataset": [
        "Baseline Full Prepared",
        "Advanced Full Prepared",
        "Baseline Train Scaled",
        "Baseline Validation Scaled",
        "Baseline Test Scaled",
        "Baseline Train Unscaled",
        "Baseline Validation Unscaled",
        "Baseline Test Unscaled",
        "Advanced Train Scaled",
        "Advanced Validation Scaled",
        "Advanced Test Scaled",
        "Advanced Train Unscaled",
        "Advanced Validation Unscaled",
        "Advanced Test Unscaled"
    ],

    "Rows": [
        prepared_train_df.shape[0],
        advanced_train_df.shape[0],
        X_train_baseline_scaled.shape[0],
        X_validation_baseline_scaled.shape[0],
        X_test_baseline_scaled.shape[0],
        X_train_baseline_unscaled.shape[0],
        X_validation_baseline_unscaled.shape[0],
        X_test_baseline_unscaled.shape[0],
        X_train_advanced_scaled.shape[0],
        X_validation_advanced_scaled.shape[0],
        X_test_advanced_scaled.shape[0],
        X_train_advanced_unscaled.shape[0],
        X_validation_advanced_unscaled.shape[0],
        X_test_advanced_unscaled.shape[0]
    ],

    "Features": [
        prepared_train_df.shape[1],
        advanced_train_df.shape[1],
        X_train_baseline_scaled.shape[1],
        X_validation_baseline_scaled.shape[1],
        X_test_baseline_scaled.shape[1],
        X_train_baseline_unscaled.shape[1],
        X_validation_baseline_unscaled.shape[1],
        X_test_baseline_unscaled.shape[1],
        X_train_advanced_scaled.shape[1],
        X_validation_advanced_scaled.shape[1],
        X_test_advanced_scaled.shape[1],
        X_train_advanced_unscaled.shape[1],
        X_validation_advanced_unscaled.shape[1],
        X_test_advanced_unscaled.shape[1]
    ]
})
final_dataset_inventory

,Dataset,Rows,Features
0,Baseline Full Prepared,307511,137
1,Advanced Full Prepared,307511,203
2,Baseline Train Scaled,34754,234
3,Baseline Validation Scaled,46127,234
4,Baseline Test Scaled,46127,234
5,Baseline Train Unscaled,34754,234
6,Baseline Validation Unscaled,46127,234
7,Baseline Test Unscaled,46127,234
8,Advanced Train Scaled,34754,298
9,Advanced Validation Scaled,46127,298


12.2 Baseline Dataset Summary

Specific Baseline Summary:

prepared_train_df

→ application_train

→ cleaning

→ anomaly treatment

→ feature engineering

→ deterministic feature selection

→ no auxiliary tables

In [263]:
baseline_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns in prepared dataframe",
        "Predictors before preprocessing",
        "Predictors after scaled preprocessing",
        "Predictors after unscaled preprocessing",
        "Train rows",
        "Validation rows",
        "Test rows"
    ],

    "Value": [
        prepared_train_df.shape[0],
        prepared_train_df.shape[1],
        X_train_baseline_selected.shape[1],
        X_train_baseline_scaled.shape[1],
        X_train_baseline_unscaled.shape[1],
        X_train_baseline_scaled.shape[0],
        X_validation_baseline_scaled.shape[0],
        X_test_baseline_scaled.shape[0]
    ]
})
baseline_summary

,Metric,Value
0,Rows,307511
1,Columns in prepared dataframe,137
2,Predictors before preprocessing,111
3,Predictors after scaled preprocessing,234
4,Predictors after unscaled preprocessing,234
5,Train rows,34754
6,Validation rows,46127
7,Test rows,46127


12.3 Advanced Dataset Summary

In [264]:
advanced_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns in prepared dataframe",
        "Auxiliary features added",
        "Predictors before preprocessing",
        "Predictors after scaled preprocessing",
        "Predictors after unscaled preprocessing",
        "Train rows",
        "Validation rows",
        "Test rows"
    ],

    "Value": [
        advanced_train_df.shape[0],
        advanced_train_df.shape[1],
        (advanced_train_df.shape[1] - prepared_train_df.shape[1]),
        X_train_advanced_selected.shape[1],
        X_train_advanced_scaled.shape[1],
        X_train_advanced_unscaled.shape[1],
        X_train_advanced_scaled.shape[0],
        X_validation_advanced_scaled.shape[0],
        X_test_advanced_scaled.shape[0]
    ]
})
advanced_summary

,Metric,Value
0,Rows,307511
1,Columns in prepared dataframe,203
2,Auxiliary features added,66
3,Predictors before preprocessing,175
4,Predictors after scaled preprocessing,298
5,Predictors after unscaled preprocessing,298
6,Train rows,34754
7,Validation rows,46127
8,Test rows,46127


12.4 Final Split Summary

In [265]:
final_split_summary = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(y_train),
        len(y_validation),
        len(y_test)
    ],
    "Percentage": [
        len(y_train) / len(split_df) * 100,
        len(y_validation) / len(split_df) * 100,
        len(y_test) / len(split_df) * 100
    ],
    "TARGET_0_Count": [
        (y_train == 0).sum(),
        (y_validation == 0).sum(),
        (y_test == 0).sum()
    ],
    "TARGET_1_Count": [
        (y_train == 1).sum(),
        (y_validation == 1).sum(),
        (y_test == 1).sum()
    ],
    "TARGET_1_Rate": [
        y_train.mean() * 100,
        y_validation.mean() * 100,
        y_test.mean() * 100
    ]
})
final_split_summary

,Split,Rows,Percentage,TARGET_0_Count,TARGET_1_Count,TARGET_1_Rate
0,Train,34754,11.301710,17377,17377,50.000000
1,Validation,46127,15.000114,42403,3724,8.073363
2,Test,46127,15.000114,42403,3724,8.073363


12.5 Final Predictor Summary

In [266]:
#Baseline
baseline_feature_type_summary = data_types[data_types["Feature"].isin(X_train_baseline_selected.columns)].groupby("Analytical Type").size().reset_index(name="Feature Count")
baseline_feature_type_summary

,Analytical Type,Feature Count
0,Binary / Flag,34
1,Categorical,11
2,External Score,6
3,Numerical Continuous,42
4,Numerical Count,14
5,Ordinal,3
6,Temporal / Days,1


In [267]:
#Advanced
advanced_feature_type_summary = data_types_advanced[data_types_advanced["Feature"].isin(X_train_advanced_selected.columns)].groupby("Analytical Type").size().reset_index(name="Feature Count")
advanced_feature_type_summary

,Analytical Type,Feature Count
0,Binary / Flag,39
1,Categorical,11
2,External Score,6
3,Numerical Continuous,88
4,Numerical Count,27
5,Ordinal,3
6,Temporal / Days,1


12.6 Data Quality Verification

In [268]:
def contains_nan(matrix):
    if sparse.issparse(matrix):
        return np.isnan(matrix.data).any()
    return np.isnan(matrix).any()


def contains_infinite(matrix):
    if sparse.issparse(matrix):
        return np.isinf(matrix.data).any()
    return np.isinf(matrix).any()

In [269]:
all_processed_datasets = {
    "Baseline Train Scaled": X_train_baseline_scaled,
    "Baseline Validation Scaled": X_validation_baseline_scaled,
    "Baseline Test Scaled": X_test_baseline_scaled,
    "Baseline Train Unscaled": X_train_baseline_unscaled,
    "Baseline Validation Unscaled": X_validation_baseline_unscaled,
    "Baseline Test Unscaled": X_test_baseline_unscaled,
    "Advanced Train Scaled": X_train_advanced_scaled,
    "Advanced Validation Scaled": X_validation_advanced_scaled,
    "Advanced Test Scaled": X_test_advanced_scaled,
    "Advanced Train Unscaled": X_train_advanced_unscaled,
    "Advanced Validation Unscaled": X_validation_advanced_unscaled,
    "Advanced Test Unscaled": X_test_advanced_unscaled
}

In [270]:
final_nan_check = []
for name, dataset in all_processed_datasets.items():
    final_nan_check.append({
        "Dataset": name,
        "Contains NaN": contains_nan(dataset)
    })
final_nan_check = pd.DataFrame(final_nan_check)
final_nan_check

,Dataset,Contains NaN
0,Baseline Train Scaled,False
1,Baseline Validation Scaled,False
2,Baseline Test Scaled,False
3,Baseline Train Unscaled,False
4,Baseline Validation Unscaled,False
5,Baseline Test Unscaled,False
6,Advanced Train Scaled,False
7,Advanced Validation Scaled,False
8,Advanced Test Scaled,False
9,Advanced Train Unscaled,False


In [271]:
#Infinite values
final_infinite_check = []
for name, dataset in all_processed_datasets.items():
    final_infinite_check.append({
        "Dataset": name,
        "Contains Infinite Values": contains_infinite(dataset)
    })
final_infinite_check = pd.DataFrame(final_infinite_check)
final_infinite_check

,Dataset,Contains Infinite Values
0,Baseline Train Scaled,False
1,Baseline Validation Scaled,False
2,Baseline Test Scaled,False
3,Baseline Train Unscaled,False
4,Baseline Validation Unscaled,False
5,Baseline Test Unscaled,False
6,Advanced Train Scaled,False
7,Advanced Validation Scaled,False
8,Advanced Test Scaled,False
9,Advanced Train Unscaled,False


In [272]:
#Duplicate applicants
baseline_duplicate_ids = prepared_train_df["SK_ID_CURR"].duplicated().sum()
advanced_duplicate_ids = advanced_train_df["SK_ID_CURR"].duplicated().sum()

print("Baseline duplicated applicants:", baseline_duplicate_ids)
print("Advanced duplicated applicants:", advanced_duplicate_ids)

Baseline duplicated applicants: 0
Advanced duplicated applicants: 0


In [273]:
#Rows preservation
print("Original application_train rows:", len(train_df))
print("Baseline prepared rows:", len(prepared_train_df))
print("Advanced prepared rows:", len(advanced_train_df))

Original application_train rows: 307511
Baseline prepared rows: 307511
Advanced prepared rows: 307511


12.7 Target Distribution Verification

In [274]:
target_distribution_summary = pd.DataFrame({
    "Dataset": [
        "Full",
        "Train",
        "Validation",
        "Test"
    ],
    "TARGET_1_Rate": [
        train_df["TARGET"].mean() * 100,
        y_train.mean() * 100,
        y_validation.mean() * 100,
        y_test.mean() * 100
    ]
})
target_distribution_summary

,Dataset,TARGET_1_Rate
0,Full,8.072882
1,Train,50.000000
2,Validation,8.073363
3,Test,8.073363


In [275]:
overall_rate = train_df["TARGET"].mean() * 100
target_distribution_summary["Difference from Overall"] = target_distribution_summary["TARGET_1_Rate"] - overall_rate
target_distribution_summary

,Dataset,TARGET_1_Rate,Difference from Overall
0,Full,8.072882,0.000000
1,Train,50.000000,41.927118
2,Validation,8.073363,0.000481
3,Test,8.073363,0.000481


## 13. Export Prepared Data

13.1 Export Strategy

I have 4 types of objects:

1. Split definitions
2. AI Studio benchmark data
3. Python modeling matrices
4. Fitted preprocessing objects

The main goal is to separate:

data/processed

→ datasets

outputs/data_preparation

→ evidence / summaries / logs

models/preprocessing

→ fitted transformers

This allows us to avoid mixing artifacts with datasets

13.2 Create Output Directories

In [276]:
processed_dir = Path("../data/processed")
splits_dir = (processed_dir / "splits")
ai_studio_dir = (processed_dir / "ai_studio")
baseline_dir = (processed_dir / "baseline")
advanced_dir = (processed_dir / "advanced")
preprocessing_dir = Path("../models/preprocessing")

In [277]:
for directory in [processed_dir, splits_dir, ai_studio_dir, baseline_dir, advanced_dir, preprocessing_dir]:
    directory.mkdir(parents=True, exist_ok=True)

13.3 Export Split IDs and Targets

In [278]:
train_ids_export = train_split_df[["SK_ID_CURR", "TARGET"]].sort_values("SK_ID_CURR").reset_index(drop=True)
validation_ids_export = validation_split_df[["SK_ID_CURR", "TARGET"]].sort_values("SK_ID_CURR").reset_index(drop=True)
test_ids_export = test_split_df[["SK_ID_CURR", "TARGET"]].sort_values("SK_ID_CURR").reset_index(drop=True)

In [279]:
train_ids_export.to_csv(splits_dir / "train_ids.csv", index=False)
validation_ids_export.to_csv(splits_dir / "validation_ids.csv", index=False)
test_ids_export.to_csv(splits_dir / "test_ids.csv", index=False)

In [280]:
#Guardar y_train, y_validation, y_test mediante conversion a dataframe
y_train_export = pd.DataFrame({"TARGET": y_train.to_numpy()})
y_validation_export = pd.DataFrame({"TARGET": y_validation.to_numpy()})
y_test_export = pd.DataFrame({"TARGET": y_test.to_numpy()})

y_train_export.to_csv(splits_dir / "y_train.csv", index=False)
y_validation_export.to_csv(splits_dir / "y_validation.csv", index=False)
y_test_export.to_csv(splits_dir / "y_test.csv", index=False)

13.4 Export AI Studio Benchmark Dataset

To keep AI Studio from becoming too complex and to demonstrate that Python outperforms the benchmark, I will use the `prepared_train_df`. Additionally, since I cannot provide the full dataset and must maintain the complexity of our splits, I will export:

- AI Studio Train

- AI Studio Validation

- AI Studio Test

In [281]:
ai_studio_train = baseline_train_df.drop(columns=["SK_ID_CURR"]).copy()
ai_studio_validation = baseline_validation_df.drop(columns=["SK_ID_CURR"]).copy()
ai_studio_test = baseline_test_df.drop(columns=["SK_ID_CURR"]).copy()

In [282]:
#Save Benchmarks
ai_studio_train.to_csv(ai_studio_dir / "ai_studio_train.csv", index=False)
ai_studio_validation.to_csv(ai_studio_dir / "ai_studio_validation.csv", index=False)
ai_studio_test.to_csv(ai_studio_dir / "ai_studio_test.csv", index=False)

13.5 Export Baseline Model Matrices

In [283]:
#Helper to save matrices
def save_matrix(matrix, filepath_without_extension):
    filepath_without_extension = Path(filepath_without_extension)

    if sparse.issparse(matrix):
        sparse.save_npz(str(filepath_without_extension) + ".npz", matrix)
        return (str(filepath_without_extension) + ".npz")
    else:
        np.save(str(filepath_without_extension) + ".npy", matrix)
        return (str(filepath_without_extension) + ".npy")

In [284]:
#Baseline Scaled
baseline_train_scaled_path = save_matrix(X_train_baseline_scaled, baseline_dir / "X_train_scaled")
baseline_validation_scaled_path = save_matrix(X_validation_baseline_scaled, baseline_dir / "X_validation_scaled")
baseline_test_scaled_path = save_matrix(X_test_baseline_scaled, baseline_dir / "X_test_scaled")

In [285]:
#Baseline Unscaled
baseline_train_unscaled_path = save_matrix(X_train_baseline_unscaled, baseline_dir / "X_train_unscaled")
baseline_validation_unscaled_path = save_matrix(X_validation_baseline_unscaled, baseline_dir / "X_validation_unscaled")
baseline_test_unscaled_path = save_matrix(X_test_baseline_unscaled, baseline_dir / "X_test_unscaled")

13.6 Export Advanced Model Matrices

In [286]:
#Advanced Scaled
advanced_train_scaled_path = save_matrix(X_train_advanced_scaled, advanced_dir / "X_train_scaled")
advanced_validation_scaled_path = save_matrix(X_validation_advanced_scaled, advanced_dir / "X_validation_scaled")
advanced_test_scaled_path = save_matrix(X_test_advanced_scaled, advanced_dir / "X_test_scaled")

In [287]:
#Advanced Unscaled
advanced_train_unscaled_path = save_matrix(X_train_advanced_unscaled, advanced_dir / "X_train_unscaled")
advanced_validation_unscaled_path = save_matrix(X_validation_advanced_unscaled, advanced_dir / "X_validation_unscaled")
advanced_test_unscaled_path = save_matrix(X_test_advanced_unscaled, advanced_dir / "X_test_unscaled")

13.7 Export Feature Names

I need to export the feature names because otherwise the matrix is not interpretable; I save them in the exact order of the features.

In [288]:
#Baseline Scaled
pd.DataFrame({"Feature": baseline_scaled_feature_names}).to_csv(baseline_dir / "scaled_feature_names.csv", index=False)

#Baseline Unscaled
pd.DataFrame({"Feature": baseline_unscaled_feature_names}).to_csv(baseline_dir / "unscaled_feature_names.csv", index=False)

#Advanced Scaled
pd.DataFrame({"Feature": advanced_scaled_feature_names}).to_csv(advanced_dir / "scaled_feature_names.csv", index=False)

#Advanced Unscales
pd.DataFrame({"Feature": advanced_unscaled_feature_names}).to_csv(advanced_dir / "unscaled_feature_names.csv", index=False)

13.8 Save Fitted Preprocessing Objects

In [289]:
joblib.dump(baseline_preprocessor_scaled, preprocessing_dir / "baseline_preprocessor_scaled.joblib")
joblib.dump(baseline_preprocessor_unscaled, preprocessing_dir / "baseline_preprocessor_unscaled.joblib")
joblib.dump(advanced_preprocessor_scaled, preprocessing_dir / "advanced_preprocessor_scaled.joblib")
joblib.dump(advanced_preprocessor_unscaled, preprocessing_dir / "advanced_preprocessor_unscaled.joblib")

['../models/preprocessing/advanced_preprocessor_unscaled.joblib']

In [290]:
#Baseline
pd.DataFrame({"Feature": baseline_features_to_drop}).to_csv(baseline_dir / "features_removed.csv", index=False)
#Advanced
pd.DataFrame({"Feature": advanced_features_to_drop}).to_csv(advanced_dir / "features_removed.csv", index=False)

In [291]:
#Save the final selection before encoding
pd.DataFrame({"Feature": X_train_baseline_selected.columns}).to_csv(baseline_dir / "selected_input_features.csv", index=False)
pd.DataFrame({"Feature": X_train_advanced_selected.columns}).to_csv(advanced_dir / "selected_input_features.csv", index=False)

13.9 Export Final Metadata

In [292]:
data_types.to_csv(baseline_dir / "data_types_baseline.csv", index=False)
data_types_advanced.to_csv(advanced_dir / "data_types_advanced.csv", index=False)

In [293]:
#Save modeling configuration
final_modelling_config = pd.DataFrame({
    "Configuration": [
        "Baseline Scaled",
        "Baseline Unscaled",
        "Advanced Scaled",
        "Advanced Unscaled"
    ],
    "Dataset": [
        "Baseline",
        "Baseline",
        "Advanced",
        "Advanced"
    ],
    "Scaling": [
        True,
        False,
        True,
        False
    ],
    "Purpose": [
        "Scale-sensitive models",
        "Tree-based models",
        "Scale-sensitive models",
        "Tree-based models"
    ],
    "Train Features": [
        X_train_baseline_scaled.shape[1],
        X_train_baseline_unscaled.shape[1],
        X_train_advanced_scaled.shape[1],
        X_train_advanced_unscaled.shape[1]
    ]
})
final_modelling_config.to_csv(processed_dir / "modelling_configurations.csv", index=False)

13.10 Create Export Manifest

In [294]:
export_manifest = pd.DataFrame({
    "Artifact": [
        "Train IDs",
        "Validation IDs",
        "Test IDs",
        "AI Studio Train",
        "AI Studio Validation",
        "AI Studio Test",
        "Baseline Scaled Matrices",
        "Baseline Unscaled Matrices",
        "Advanced Scaled Matrices",
        "Advanced Unscaled Matrices",
        "Baseline Preprocessor Scaled",
        "Baseline Preprocessor Unscaled",
        "Advanced Preprocessor Scaled",
        "Advanced Preprocessor Unscaled",
        "Baseline Feature Names",
        "Advanced Feature Names"
    ],
    "Purpose": [
        "Freeze training split",
        "Freeze validation split",
        "Freeze internal test split",
        "AI Studio benchmark training",
        "AI Studio benchmark validation",
        "AI Studio benchmark testing",
        "Scale-sensitive Python models",
        "Tree-based Python models",
        "Scale-sensitive advanced models",
        "Tree-based advanced models",
        "Reproduce scaled baseline preprocessing",
        "Reproduce unscaled baseline preprocessing",
        "Reproduce scaled advanced preprocessing",
        "Reproduce unscaled advanced preprocessing",
        "Interpret baseline model columns",
        "Interpret advanced model columns"
    ],
    "Status": ["Exported"] * 16
})

export_manifest.to_csv("../outputs/data_preparation/export_manifest.csv", index=False)

In [295]:
print("Data Preparation completed successfully.")
print("Baseline and advanced modelling datasets have been prepared and exported.")
print("Train, validation and internal test splits are fixed and reproducible.")
print("All fitted preprocessing objects were learned from the internal training set only.")
print("Datasets are ready for AI Studio benchmarking and Python modelling.")

Data Preparation completed successfully.
Baseline and advanced modelling datasets have been prepared and exported.
Train, validation and internal test splits are fixed and reproducible.
All fitted preprocessing objects were learned from the internal training set only.
Datasets are ready for AI Studio benchmarking and Python modelling.


In [296]:
print("Duplicate selected input columns:", X_train_baseline_selected.columns[X_train_baseline_selected.columns.duplicated()].tolist())
print("Duplicate baseline metadata features:", data_types.loc[data_types["Feature"].duplicated(keep=False), "Feature"].value_counts().to_dict())
print("Duplicate baseline group selectors:", {name: [feature for feature in features if features.count(feature) > 1] for name, features in baseline_groups.items()})

Duplicate selected input columns: []
Duplicate baseline metadata features: {}
Duplicate baseline group selectors: {'continuous': [], 'count': [], 'external': [], 'categorical': [], 'binary': [], 'ordinal': []}
